In [ ]:
# 如果没安装，先运行：
import scanpy as sc
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import liana as li


In [ ]:
adata_ec=sc.read_h5ad('adata_ec_clean2.h5ad') 
adata_mp=sc.read_h5ad('results/adata_macrophage_with_func_scores.h5ad') 

In [ ]:
adata_ec = adata_ec.copy()
adata_mp = adata_mp.copy()

adata_ec.obs["celltype_for_liana"] = adata_ec.obs["Final_Cell_Type"].astype(str)
adata_mp.obs["celltype_for_liana"] = adata_mp.obs["cell_type2"].astype(str)


In [ ]:
adata_ec.obs["major_group"] = "EC"
adata_mp.obs["major_group"] = "MP"


In [ ]:
print(adata_ec.shape)
print(adata_mp.shape)

print(adata_ec.var_names[:5])
print(adata_mp.var_names[:5])


In [ ]:
common_genes = adata_ec.var_names.intersection(adata_mp.var_names)

print("EC genes:", adata_ec.n_vars)
print("MP genes:", adata_mp.n_vars)
print("Common genes:", len(common_genes))


In [ ]:
import pandas as pd
import numpy as np
import scanpy as sc
import anndata as ad
from scipy import sparse
from gprofiler import GProfiler


In [ ]:
print(adata_ec.var_names[:30].tolist())
print("Pig EC genes:", adata_ec.n_vars)

print(adata_mp.var_names[:30].tolist())
print("Human MP genes:", adata_mp.n_vars)


In [ ]:
mp_genes = pd.Index(adata_mp.var_names.astype(str))

is_ensg = mp_genes.str.startswith("ENSG")

print("MP total genes:", len(mp_genes))
print("MP ENSG genes:", is_ensg.sum())
print("MP symbol-like genes:", (~is_ensg).sum())


In [ ]:
import mygene
import pandas as pd
import numpy as np
import anndata as ad
from scipy import sparse

mg = mygene.MyGeneInfo()

ensg_genes = mp_genes[is_ensg].tolist()

ensg_query = mg.querymany(
    ensg_genes,
    scopes="ensembl.gene",
    fields="symbol",
    species="human",
    as_dataframe=True
)

print(ensg_query.head())


In [ ]:
ensg_map = (
    ensg_query[["symbol"]]
    .dropna()
    .reset_index()
    .rename(columns={"query": "ensembl_gene", "symbol": "human_symbol"})
)

ensg_map = ensg_map.drop_duplicates("ensembl_gene")

print(ensg_map.head())
print("Mapped ENSG:", ensg_map["ensembl_gene"].nunique())


In [ ]:
ensg_to_symbol = dict(
    zip(ensg_map["ensembl_gene"], ensg_map["human_symbol"])
)

new_mp_genes = []

for g in adata_mp.var_names.astype(str):
    if g.startswith("ENSG") and g in ensg_to_symbol:
        new_mp_genes.append(ensg_to_symbol[g])
    elif g.startswith("ENSG") and g not in ensg_to_symbol:
        new_mp_genes.append(None)
    else:
        new_mp_genes.append(g)

adata_mp_symbol = adata_mp.copy()
adata_mp_symbol.var["original_gene"] = adata_mp_symbol.var_names.astype(str)
adata_mp_symbol.var["human_symbol"] = new_mp_genes


In [ ]:
keep = adata_mp_symbol.var["human_symbol"].notna().values

adata_mp_symbol = adata_mp_symbol[:, keep].copy()
adata_mp_symbol.var_names = adata_mp_symbol.var["human_symbol"].astype(str)


In [ ]:
def collapse_duplicate_genes(adata, gene_col=None):
    """
    Collapse duplicated genes by summing expression across duplicated columns.
    """
    if gene_col is None:
        genes = pd.Index(adata.var_names.astype(str))
    else:
        genes = pd.Index(adata.var[gene_col].astype(str))

    gene_df = pd.DataFrame({
        "gene": genes,
        "idx": np.arange(len(genes))
    })

    unique_genes = gene_df["gene"].drop_duplicates().tolist()

    cols = []
    X = adata.X

    for gene in unique_genes:
        idx = gene_df.loc[gene_df["gene"] == gene, "idx"].values
        sub_x = X[:, idx]

        if sparse.issparse(sub_x):
            summed = sub_x.sum(axis=1)
            cols.append(sparse.csr_matrix(summed))
        else:
            summed = sub_x.sum(axis=1, keepdims=True)
            cols.append(sparse.csr_matrix(summed))

    X_new = sparse.hstack(cols).tocsr()

    adata_new = ad.AnnData(
        X=X_new,
        obs=adata.obs.copy(),
        var=pd.DataFrame(index=unique_genes)
    )

    return adata_new


In [ ]:
adata_mp_symbol = collapse_duplicate_genes(adata_mp_symbol)

print("MP genes after symbol cleanup:", adata_mp_symbol.n_vars)
print(adata_mp_symbol.var_names[:30].tolist())


In [ ]:
from pybiomart import Server
import pandas as pd

server = Server(host="http://www.ensembl.org")

pig_dataset = server.marts["ENSEMBL_MART_ENSEMBL"].datasets["sscrofa_gene_ensembl"]

orth = pig_dataset.query(
    attributes=[
        "external_gene_name",
        "hsapiens_homolog_associated_gene_name",
        "hsapiens_homolog_orthology_type",
        "hsapiens_homolog_orthology_confidence"
    ]
)

orth.columns = [
    "pig_gene",
    "human_gene",
    "orthology_type",
    "orthology_confidence"
]

print(orth.head())
print(orth.shape)


In [ ]:
orth_map = orth.dropna(subset=["pig_gene", "human_gene"]).copy()

orth_map = orth_map[
    (orth_map["pig_gene"].astype(str).str.len() > 0) &
    (orth_map["human_gene"].astype(str).str.len() > 0)
].copy()

# 优先保留 one2one；如果太少，再放宽
orth_map_one2one = orth_map[
    orth_map["orthology_type"] == "ortholog_one2one"
].copy()

print("All mapped pig genes:", orth_map["pig_gene"].nunique())
print("All human ortholog genes:", orth_map["human_gene"].nunique())
print("One2one mapped pig genes:", orth_map_one2one["pig_gene"].nunique())
print("One2one human ortholog genes:", orth_map_one2one["human_gene"].nunique())


In [ ]:
orth_map_use = orth_map_one2one.copy()

# 如果有 orthology confidence 列，优先保留高置信
if "orthology_confidence" in orth_map_use.columns:
    orth_map_use = orth_map_use[
        orth_map_use["orthology_confidence"].astype(str).isin(["1", "1.0"])
    ].copy()

print("Using mapped pig genes:", orth_map_use["pig_gene"].nunique())
print("Using human ortholog genes:", orth_map_use["human_gene"].nunique())


In [ ]:
pig_to_human = dict(
    zip(orth_map_use["pig_gene"], orth_map_use["human_gene"])
)

mapped_pig_genes = [
    g for g in adata_ec.var_names.astype(str)
    if g in pig_to_human
]

adata_ec_mapped = adata_ec[:, mapped_pig_genes].copy()

adata_ec_mapped.var["pig_gene"] = adata_ec_mapped.var_names.astype(str)
adata_ec_mapped.var["human_symbol"] = [
    pig_to_human[g] for g in adata_ec_mapped.var_names.astype(str)
]

adata_ec_human = collapse_duplicate_genes(
    adata_ec_mapped,
    gene_col="human_symbol"
)

print("Pig EC original genes:", adata_ec.n_vars)
print("Pig EC mapped genes before collapse:", adata_ec_mapped.n_vars)
print("Pig EC mapped to human genes:", adata_ec_human.n_vars)
print(adata_ec_human.var_names[:30].tolist())


In [ ]:
common_genes = adata_ec_human.var_names.intersection(
    adata_mp_symbol.var_names
)

print("Pig EC human ortholog genes:", adata_ec_human.n_vars)
print("Human MP symbol genes:", adata_mp_symbol.n_vars)
print("Common genes after ortholog cleanup:", len(common_genes))


In [ ]:
key_lr_genes = [
    "VEGFA", "KDR", "FLT1",
    "TGFB1", "TGFBR1", "TGFBR2",
    "TNF", "TNFRSF1A", "TNFRSF1B",
    "IL1B", "IL1R1", "IL1R2",
    "CCL2", "CCR2",
    "CXCL12", "CXCR4",
    "ICAM1", "ITGAL", "ITGB2",
    "VCAM1", "ITGA4", "ITGB1",
    "SELE", "SELPLG",
    "SPP1", "CD44",
    "THBS1", "THBS2",
    "PLG", "PLAUR"
]

coverage = pd.DataFrame({
    "gene": key_lr_genes,
    "in_pigEC_as_human": [g in adata_ec_human.var_names for g in key_lr_genes],
    "in_humanMP": [g in adata_mp_symbol.var_names for g in key_lr_genes],
    "in_common": [g in common_genes for g in key_lr_genes]
})

print(coverage)


In [ ]:
print("EC:", adata_ec)
print("EC n_vars:", adata_ec.n_vars)
print("EC raw:", adata_ec.raw)
print("EC layers:", list(adata_ec.layers.keys()))
print("EC var columns:", adata_ec.var.columns.tolist())

print("MP:", adata_mp_symbol)
print("MP n_vars:", adata_mp_symbol.n_vars)
print("MP raw:", adata_mp_symbol.raw)
print("MP layers:", list(adata_mp_symbol.layers.keys()))
print("MP var columns:", adata_mp_symbol.var.columns.tolist())


In [ ]:
print(adata_ec.var.head())
print(adata_ec.var.tail())

for col in adata_ec.var.columns:
    if any(x in col.lower() for x in ["highly", "hvg", "marker", "feature", "gene"]):
        print(col, adata_ec.var[col].head().tolist())


In [ ]:
import pandas as pd
import anndata as ad

adata_ec_full = adata_ec.raw.to_adata()
adata_ec_full.obs = adata_ec.obs.copy()

print("EC current genes:", adata_ec.n_vars)
print("EC raw genes:", adata_ec_full.n_vars)
print("MP genes:", adata_mp_symbol.n_vars)


In [ ]:
# 使用 one2one ortholog
orth_map_use = orth_map_one2one.copy()

# 如果有 confidence 列，保留高置信；如果没有也不影响
if "orthology_confidence" in orth_map_use.columns:
    orth_map_use = orth_map_use[
        orth_map_use["orthology_confidence"].astype(str).isin(["1", "1.0"])
    ].copy()

pig_to_human = dict(
    zip(
        orth_map_use["pig_gene"].astype(str),
        orth_map_use["human_gene"].astype(str)
    )
)

mapped_pig_genes = [
    g for g in adata_ec_full.var_names.astype(str)
    if g in pig_to_human
]

adata_ec_mapped = adata_ec_full[:, mapped_pig_genes].copy()

adata_ec_mapped.var["pig_gene"] = adata_ec_mapped.var_names.astype(str)
adata_ec_mapped.var["human_symbol"] = [
    pig_to_human[g] for g in adata_ec_mapped.var_names.astype(str)
]

print("Pig EC raw mapped genes before collapse:", adata_ec_mapped.n_vars)
print("Unique human symbols before collapse:", adata_ec_mapped.var["human_symbol"].nunique())


In [ ]:
import numpy as np
from scipy import sparse
import anndata as ad

def collapse_duplicate_genes(adata, gene_col):
    gene_names = adata.var[gene_col].astype(str).values
    unique_genes = pd.Index(pd.unique(gene_names))

    gene_to_idx = pd.Series(
        np.arange(len(unique_genes)),
        index=unique_genes
    )

    col_idx = gene_to_idx[gene_names].values

    mapping = sparse.csr_matrix(
        (
            np.ones(len(col_idx)),
            (np.arange(len(col_idx)), col_idx)
        ),
        shape=(adata.n_vars, len(unique_genes))
    )

    X_collapsed = adata.X @ mapping

    out = ad.AnnData(
        X=X_collapsed,
        obs=adata.obs.copy(),
        var=pd.DataFrame(index=unique_genes)
    )

    return out


In [ ]:
adata_ec_human = collapse_duplicate_genes(
    adata_ec_mapped,
    gene_col="human_symbol"
)

common_genes = adata_ec_human.var_names.intersection(
    adata_mp_symbol.var_names
)

print("Pig EC raw mapped to human genes:", adata_ec_human.n_vars)
print("Human MP symbol genes:", adata_mp_symbol.n_vars)
print("Common genes after raw ortholog cleanup:", len(common_genes))


In [ ]:
key_lr_genes = [
    "VEGFA", "KDR", "FLT1",
    "TGFB1", "TGFBR1", "TGFBR2",
    "TNF", "TNFRSF1A", "TNFRSF1B",
    "IL1B", "IL1R1", "IL1R2",
    "CCL2", "CCR2",
    "CXCL12", "CXCR4",
    "ICAM1", "ITGAL", "ITGB2",
    "VCAM1", "ITGA4", "ITGB1",
    "SELE", "SELPLG",
    "SPP1", "CD44",
    "THBS1", "THBS2",
    "PLG", "PLAUR"
]

coverage = pd.DataFrame({
    "gene": key_lr_genes,
    "in_pigEC_as_human": [g in adata_ec_human.var_names for g in key_lr_genes],
    "in_humanMP": [g in adata_mp_symbol.var_names for g in key_lr_genes],
    "in_common": [g in common_genes for g in key_lr_genes]
})

print(coverage)
print("Key LR genes in common:", coverage["in_common"].sum(), "/", len(coverage))


In [ ]:
import anndata as ad
import pandas as pd

adata_ec_use = adata_ec_human[:, common_genes].copy()
adata_mp_use = adata_mp_symbol[:, common_genes].copy()

adata_ec_use.obs["cell_group"] = "EC_total"
adata_mp_use.obs["cell_group"] = "MP_total"

adata_ec_use.obs["dataset_group"] = "Pig_EC_mapped_to_human"
adata_mp_use.obs["dataset_group"] = "Human_MP"

adata_cross = ad.concat(
    {
        "PigEC_asHuman": adata_ec_use,
        "HumanMP": adata_mp_use
    },
    axis=0,
    join="inner",
    label="dataset",
    index_unique="-"
)

print(adata_cross)
print(pd.crosstab(
    adata_cross.obs["time_point"],
    adata_cross.obs["cell_group"]
))


In [ ]:
import liana as li
import pandas as pd

main_time_points = ["D14", "D33", "D61"]

adata_main = adata_cross[
    adata_cross.obs["time_point"].isin(main_time_points)
].copy()

results_by_time = {}

for tp in main_time_points:
    print(f"\nRunning LIANA for {tp}...")

    adata_tp = adata_main[
        adata_main.obs["time_point"] == tp
    ].copy()

    print(pd.crosstab(
        adata_tp.obs["time_point"],
        adata_tp.obs["cell_group"]
    ))

    li.mt.rank_aggregate(
        adata_tp,
        groupby="cell_group",
        resource_name="consensus",
        expr_prop=0.1,
        min_cells=10,
        use_raw=False,
        verbose=True
    )

    res = adata_tp.uns["liana_res"].copy()
    res["time_point"] = tp
    results_by_time[tp] = res

liana_cross_res = pd.concat(
    results_by_time.values(),
    ignore_index=True
)

ec_mp_cross_res = liana_cross_res[
    (
        (liana_cross_res["source"] == "EC_total") &
        (liana_cross_res["target"] == "MP_total")
    ) |
    (
        (liana_cross_res["source"] == "MP_total") &
        (liana_cross_res["target"] == "EC_total")
    )
].copy()

ec_mp_cross_res["direction"] = (
    ec_mp_cross_res["source"] + "_to_" + ec_mp_cross_res["target"]
)

ec_mp_cross_res.to_csv(
    "PigEC_HumanMP_ortholog_liana_results.csv",
    index=False
)

print(ec_mp_cross_res.shape)
print(ec_mp_cross_res.head())


In [ ]:
print(ec_mp_cross_res.shape)
print(ec_mp_cross_res.columns.tolist())
print(ec_mp_cross_res.head())


In [ ]:
score_cols = [
    "time_point", "direction", "source", "target",
    "ligand_complex", "receptor_complex",
    "cellphone_pvals", "lrscore", "lr_means",
    "expr_prod", "scaled_weight", "lr_logfc",
    "spec_weight", "specificity_rank", "magnitude_rank"
]

existing_score_cols = [
    c for c in score_cols
    if c in ec_mp_cross_res.columns
]

top_liana = (
    ec_mp_cross_res
    .sort_values(
        ["time_point", "cellphone_pvals", "lrscore"],
        ascending=[True, True, False],
        na_position="last"
    )
    [existing_score_cols]
)

print(top_liana.head(80))


In [ ]:
qc_liana = (
    ec_mp_cross_res
    .assign(
        p_lt_005=ec_mp_cross_res["cellphone_pvals"] < 0.05,
        p_lt_01=ec_mp_cross_res["cellphone_pvals"] < 0.1,
        has_lrscore=ec_mp_cross_res["lrscore"].notna()
    )
    .groupby(["time_point", "direction"])
    .agg(
        n_lr=("ligand_complex", "count"),
        n_p_lt_005=("p_lt_005", "sum"),
        n_p_lt_01=("p_lt_01", "sum"),
        n_lrscore=("has_lrscore", "sum"),
        median_p=("cellphone_pvals", "median"),
        max_lrscore=("lrscore", "max")
    )
    .reset_index()
)

print(qc_liana)


In [ ]:
ec_mp_sig = ec_mp_cross_res[
    (ec_mp_cross_res["cellphone_pvals"] < 0.05) &
    (ec_mp_cross_res["lrscore"].notna())
].copy()

ec_mp_sig = ec_mp_sig.sort_values(
    ["time_point", "cellphone_pvals", "lrscore"],
    ascending=[True, True, False]
)

ec_mp_sig.to_csv(
    "PigEC_HumanMP_ortholog_liana_significant_LR.csv",
    index=False
)

print(ec_mp_sig.shape)
print(ec_mp_sig[existing_score_cols].head(100))

In [ ]:
print(qc_liana)
print(ec_mp_sig.shape)
print(ec_mp_sig[existing_score_cols].head(100))

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from matplotlib.ticker import MaxNLocator

# =========================================================
# 1. 从已有的全局显著结果 ec_mp_sig 中构建 df_counts
# =========================================================
print("1. 正在从全局显著结果中统计各时间点的通讯数量...")

# 📌 请根据你数据中大类的实际名字进行修改，如果是 "Human_MP"/"Pig_EC" 或 "MP"/"EC"
# 这里默认使用常见的大类命名
mp_cell_name = "MP_total"
ec_cell_name = "EC_total"

# 定义两个方向的筛选条件
mp_to_ec_mask = (ec_mp_sig['source'] == mp_cell_name) & (ec_mp_sig['target'] == ec_cell_name)
ec_to_mp_mask = (ec_mp_sig['source'] == ec_cell_name) & (ec_mp_sig['target'] == mp_cell_name)

# 筛选出属于这两种跨物种通讯的数据
ec_mp_sig_cross = ec_mp_sig[mp_to_ec_mask | ec_to_mp_mask].copy()

# 标记方向标签
ec_mp_sig_cross['Direction'] = np.where(
    ec_mp_sig_cross['source'] == mp_cell_name,
    "Human MP $\\rightarrow$ Pig EC",
    "Pig EC $\\rightarrow$ Human MP"
)

# 分组计数
df_counts_raw = ec_mp_sig_cross.groupby(['time_point', 'Direction']).size().reset_index(name='Significant_Interactions')

# 补全缺失的时间点和方向组合，防止高度为0时缺柱子
tps = ["D14", "D33", "D61"]
dirs = ["Human MP $\\rightarrow$ Pig EC", "Pig EC $\\rightarrow$ Human MP"]
full_idx = pd.MultiIndex.from_product([tps, dirs], names=['time_point', 'Direction'])
df_counts = df_counts_raw.set_index(['time_point', 'Direction']).reindex(full_idx, fill_value=0).reset_index()


# =========================================================
# 2. 图A：绘制各时间点通讯数量对比图 (Barplot - 提取自热图的蓝色系)
# =========================================================
print("2. 正在绘制通讯方向对比柱状图...")

# 确保时间点按照正确的时序排列
df_counts["time_point"] = pd.Categorical(df_counts["time_point"], categories=["D14", "D33", "D61"], ordered=True)

# 开始画图
plt.figure(figsize=(4.5, 4.5))
sns.set_theme(style="ticks")

# 使用从您热图中精准提取的蓝色和深蓝色
palette_colors = {
    "Human MP $\\rightarrow$ Pig EC": "#08457E",  # 深蓝色
    "Pig EC $\\rightarrow$ Human MP": "#4B9CD3"   # 中蓝色
}

ax = sns.barplot(
    data=df_counts,
    x="time_point",
    y="Significant_Interactions",
    hue="Direction",
    palette=palette_colors,
    edgecolor="#333333",                         # 深灰边框
    linewidth=1.0                                # 细化边框线
)

plt.title("Number of Significant Cross-species Interactions", fontsize=12, fontweight='bold', pad=15)
plt.ylabel("Significant LR Pairs (p < 0.05)", fontsize=11)
plt.xlabel("Time Point", fontsize=11)

# 美化图例
plt.legend(
    title="Direction", 
    bbox_to_anchor=(1.02, 1), 
    loc='upper left',
    frameon=False,
    fontsize=10,
    title_fontsize=11
)

sns.despine()

# 限制 Y 轴刻度为整数
ax.yaxis.set_major_locator(MaxNLocator(integer=True))

plt.tight_layout()
plt.savefig("FigA_Interaction_Counts_Barplot_Blue.pdf", dpi=300, bbox_inches="tight")
plt.show()
print("-> 已保存学术蓝色系柱状图：FigA_Interaction_Counts_Barplot_Blue.pdf")


In [ ]:
# =========================================================
# 图A：绘制各时间点通讯数量对比图 (Barplot - 提取自热图的蓝色系)
# =========================================================
print("2. 正在绘制通讯方向对比柱状图...")

# 确保时间点按照正确的时序排列
df_counts["time_point"] = pd.Categorical(df_counts["time_point"], categories=["D14", "D33", "D61"], ordered=True)

# 开始画图
plt.figure(figsize=(4.5, 4.5))
sns.set_theme(style="ticks")

# 使用从您热图中精准提取的蓝色和深蓝色
palette_colors = {
    "Human MP $\\rightarrow$ Pig EC": "#08457E", # 中蓝色 (Medium Blue)
    "Pig EC $\\rightarrow$ Human MP": "#4B9CD3"  # 深蓝色 (Deep Royal Blue)
}

ax = sns.barplot(
    data=df_counts,
    x="time_point",
    y="Significant_Interactions",
    hue="Direction",
    palette=palette_colors,
    edgecolor="#333333",                         # 深灰边框
    linewidth=1.0                                # 细化边框线
)

plt.title("Number of Significant Cross-species Interactions", fontsize=12, fontweight='bold', pad=15)
plt.ylabel("Significant LR Pairs (p < 0.05)", fontsize=11)
plt.xlabel("Time Point", fontsize=11)

# 美化图例
plt.legend(
    title="Direction", 
    bbox_to_anchor=(1.02, 1), 
    loc='upper left',
    frameon=False,
    fontsize=10,
    title_fontsize=11
)

sns.despine()

# 限制 Y 轴刻度为整数
from matplotlib.ticker import MaxNLocator
ax.yaxis.set_major_locator(MaxNLocator(integer=True))

plt.tight_layout()
plt.savefig("FigA_Interaction_Counts_Barplot_Blue.pdf", dpi=300, bbox_inches="tight")
plt.show()
print("-> 已保存学术蓝色系柱状图：FigA_Interaction_Counts_Barplot_Blue.pdf")


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# =========================================================
# 3. 总体通讯：信号通路归类与时序准备
# =========================================================
print("3. 正在对总体 MP 和总体 EC 的显著通讯进行信号通路分类...")

# 确保时间点时序正确
ec_mp_sig["time_point"] = pd.Categorical(ec_mp_sig["time_point"], categories=["D14", "D33", "D61"], ordered=True)
ec_mp_sig["LR_pair"] = ec_mp_sig["ligand_complex"] + " $\\rightarrow$ " + ec_mp_sig["receptor_complex"]

# 精细化定制通路词典（极度贴合您导出的 CSV 中出现的分子如 C3, CXCL8, PDGFC, CALM3, SEMA3B, TGFB1, ICAM1 等）
def get_pathway_group(lr_pair):
    pair_upper = lr_pair.upper()
    pathway_rules = {
        "Growth Factor / Angiogenesis": ["PDGF", "EGF", "IGF", "FGF", "VEGF", "FLT", "KDR", "NAMPT", "IGF1R"],
        "TGF-b / BMP Signaling": ["TGFB", "ENG", "INHBC", "ACVR", "BMP"],
        "Chemokine / Cytokine": ["CXCL", "CCL", "CXCR", "CCR", "ACKR"],
        "Complement System": ["C3", "C5", "C3AR", "C5AR", "CFB", "CD46"],
        "Semaphorin Guidance": ["SEMA", "NRP1", "NRP2", "PLXN"],
        "Adhesion / ECM / Matrix": ["CDH", "VCAN", "FN1", "TLR", "ITG", "ICAM", "VCAM", "PECAM", "MMP", "LRP", "PLAUR", "THBS"],
        "Metabolic / Calcium / Others": ["CALM", "EGFR", "ESR1", "RYR", "CACNA", "ENTPD1", "ADORA", "INSR", "F8", "APOE"]
    }
    for pathway, keywords in pathway_rules.items():
        if any(kw in pair_upper for kw in keywords):
            return pathway
    return "Other Pathways"

ec_mp_sig["Pathway"] = ec_mp_sig["LR_pair"].apply(get_pathway_group)

# 如果您的数据中 source/target 表示方式不同，自动识别：
# 常见可能为：'MP_total', 'EC_total' 或 'Human MP', 'Pig EC'
mp_labels = [c for c in ec_mp_sig["source"].unique() if "MP" in str(c).upper() or "MACROPHAGE" in str(c).upper()]
ec_labels = [c for c in ec_mp_sig["target"].unique() if "EC" in str(c).upper() or "ENDOTHELIAL" in str(c).upper()]

# 区分通信方向
# 方向1：MP发送给EC (MP -> EC)
mp_to_ec_total = ec_mp_sig[
    ec_mp_sig["source"].isin(mp_labels) & ec_mp_sig["target"].isin(ec_labels)
].copy() if (len(mp_labels) > 0 and len(ec_labels) > 0) else ec_mp_sig[ec_mp_sig["Direction"].str.contains("MP.*EC|MP_total.*EC_total", regex=True, na=False)].copy()

# 方向2：EC发送给MP (EC -> MP)
ec_to_mp_total = ec_mp_sig[
    ec_mp_sig["source"].isin(ec_labels) & ec_mp_sig["target"].isin(mp_labels)
].copy() if (len(mp_labels) > 0 and len(ec_labels) > 0) else ec_mp_sig[ec_mp_sig["Direction"].str.contains("EC.*MP|EC_total.*MP_total", regex=True, na=False)].copy()

print(f"-> 分类完成！MP -> EC 共有 {len(mp_to_ec_total)} 条记录；EC -> MP 共有 {len(ec_to_mp_total)} 条记录。")


In [ ]:
# =========================================================
# 方案一：总体通路时序热图 (Heatmap - 缺失值留白)
# =========================================================
print("正在绘制总体通路时序热图...")

# 1. 重新计算均值，但不填充 0
trend_df = mp_to_ec_total.groupby(["time_point", "Pathway"], observed=True)["lrscore"].mean().reset_index()
trend_df = trend_df[trend_df["Pathway"] != "Other Pathways"].copy()

# 2. 数据透视
heatmap_data = trend_df.pivot(index="Pathway", columns="time_point", values="lrscore")

# 3. 绘图
plt.figure(figsize=(5, 4.5))
sns.set_theme(style="white")

# 使用 cbar_kws 调整图例，mask 会自动让 NaN (无数据) 显示为背景色（或灰色）
ax = sns.heatmap(
    heatmap_data, 
    cmap="Blues", 
    annot=True,            # 在格子内显示具体平均分，增强可读性
    fmt=".2f",
    linewidths=1.0,        
    linecolor='white',
    cbar_kws={"shrink": .7, "label": "Mean LR Score"},
    mask=heatmap_data.isnull(), # 自动隐藏无数据的格子
    vmin=0.3                    # 设定合理的颜色起点
)

# 将无数据的格子背景设为浅灰色，代表未检测到 (Not Detected)
ax.set_facecolor('#EAEAEA') 

# 加上外边框
for _, spine in ax.spines.items():
    spine.set_visible(True)
    spine.set_color('black')
    spine.set_linewidth(1.2)

plt.title("Macro-Pathway Activity Over Time\nTotal MP $\\rightarrow$ Total EC", fontsize=12, fontweight='bold', pad=15)
plt.xlabel("Time Point", fontsize=11, fontweight='bold')
plt.ylabel("Signaling Family", fontsize=11, fontweight='bold')
plt.xticks(fontsize=11)
plt.yticks(fontsize=10.5)

plt.tight_layout()
plt.savefig("FigC_Total_MP_to_EC_Pathway_Heatmap.pdf", dpi=300, bbox_inches="tight")
plt.show()


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# =========================================================
# 1. 数据准备与排序（确保通路从上到下与热图一致）
# =========================================================
pathway_order = [
    "TGF-b / BMP Signaling",
    "Chemokine / Cytokine",
    "Semaphorin Guidance",
    "Growth Factor / Angiogenesis",
    "Adhesion / ECM / Matrix",
    "Complement System",
    "Metabolic / Calcium / Others"
]

# 筛选对应通路数据
sub_df = mp_to_ec_total[mp_to_ec_total["Pathway"].isin(pathway_order)].copy()

# 筛选每个通路最显著的 4 个对
top_pairs = (
    sub_df.groupby(["Pathway", "LR_pair"])["lrscore"]
    .mean()
    .reset_index()
    .sort_values(["Pathway", "lrscore"], ascending=[True, False])
    .groupby("Pathway")
    .head(4)
)

plot_data = sub_df[sub_df["LR_pair"].isin(top_pairs["LR_pair"])].copy()

# 计算唯一的 (Pathway, LR_pair) 组合并映射到固定的 Y 轴坐标
unique_rows = plot_data[["Pathway", "LR_pair"]].drop_duplicates().copy()
unique_rows["Pathway"] = pd.Categorical(unique_rows["Pathway"], categories=pathway_order, ordered=True)
# 排序：按通路顺序正向排序，然后倒序，使 pathway_order 中的第一个通路画在最上方
unique_rows = unique_rows.sort_values(["Pathway", "LR_pair"], ascending=[True, True])
unique_rows = unique_rows.iloc[::-1].reset_index(drop=True)
unique_rows["y_idx"] = unique_rows.index

# 合并 Y 坐标回主数据
plot_data = plot_data.merge(unique_rows, on=["Pathway", "LR_pair"])

# =========================================================
# 2. 开始画图
# =========================================================
fig, ax = plt.subplots(figsize=(6, 9.5))
sns.set_theme(style="ticks")

# 设定 X 轴范围（D14, D33, D61 对应 0, 1, 2）
ax.set_xlim(-0.4, 2.4)
ax.set_ylim(-0.6, len(unique_rows) - 0.4)

# 绘制交替背景色带 (Shading)
pathways = unique_rows["Pathway"].unique()
for idx, pw in enumerate(pathways):
    pw_rows = unique_rows[unique_rows["Pathway"] == pw]
    min_y = pw_rows["y_idx"].min()
    max_y = pw_rows["y_idx"].max()
    
    # 奇数行填充极浅灰色，偶数行保留白色
    if idx % 2 == 1:
        ax.axhspan(min_y - 0.5, max_y + 0.5, facecolor='#F2F4F4', alpha=0.6, zorder=0)

# 绘制通路之间的硬分割线
for i in range(len(unique_rows) - 1):
    if unique_rows.loc[i, "Pathway"] != unique_rows.loc[i+1, "Pathway"]:
        ax.axhline(y=i+0.5, color="#BDC3C7", linestyle="-", linewidth=1.0, zorder=1)

# 绘制气泡图
scatter = ax.scatter(
    x=plot_data["time_point"],
    y=plot_data["y_idx"],
    s=plot_data["lrscore"] * 450,    # 气泡大小比例
    c=plot_data["lrscore"],          # 气泡颜色映射
    cmap="Blues",                    # 经典学术蓝
    edgecolors="#1A252C",            # 深色描边
    linewidths=0.8,
    zorder=3                         # 确保气泡在背景色带上方
)

# =========================================================
# 3. 绘制右侧学术大括号与通路名称
# =========================================================
for pw in pathways:
    pw_rows = unique_rows[unique_rows["Pathway"] == pw]
    min_y = pw_rows["y_idx"].min()
    max_y = pw_rows["y_idx"].max()
    mid_y = (min_y + max_y) / 2.0
    
    # 括号绘制坐标 (放在绘图区右侧边缘)
    x_bracket = 2.50
    x_tick = 2.45
    y_start = min_y - 0.35
    y_end = max_y + 0.35
    
    # 画括号的竖线和上下横突
    ax.plot([x_bracket, x_bracket], [y_start, y_end], color="#5D6D7E", linewidth=1.5, clip_on=False)
    ax.plot([x_tick, x_bracket], [y_start, y_start], color="#5D6D7E", linewidth=1.5, clip_on=False)
    ax.plot([x_tick, x_bracket], [y_end, y_end], color="#5D6D7E", linewidth=1.5, clip_on=False)
    
    # 在括号右侧写上通路名字
    ax.text(
        x_bracket + 0.08, 
        mid_y, 
        pw, 
        va='center', 
        ha='left', 
        fontsize=10.5, 
        fontweight='bold', 
        color='#2C3E50',
        clip_on=False
    )

# =========================================================
# 4. 图表细节美化
# =========================================================
# 设置 Y 轴刻度标签（只保留配体-受体名）
ax.set_yticks(unique_rows["y_idx"])
ax.set_yticklabels(unique_rows["LR_pair"], fontsize=10.5)

# 设置 X 轴
ax.set_xlabel("Time Point (Days Post-transplantation)", fontsize=11, fontweight='bold', labelpad=10)
plt.xticks(fontsize=11, fontweight='bold')
ax.grid(axis='x', linestyle=':', alpha=0.5, color='#BDC3C7') # 仅保留垂直辅助线

# 标题
plt.title("Key Ligand-Receptor Pairs Driving Pathway Dynamics\nTotal MP $\\rightarrow$ Total EC", 
          fontsize=12.5, fontweight='bold', pad=20)

# 移除上、右、左边框，保留底边框
sns.despine(top=True, right=True, left=False, bottom=False)

# =========================================================
# 5. 图例设置（放置在图表正下方）
# =========================================================
# 提取气泡的大小和颜色信息创建图例
handles, labels = scatter.legend_elements(prop="sizes", alpha=0.6, num=5, func=lambda x: x/450)
legend2 = ax.legend(
    handles, 
    labels, 
    loc="upper center", 
    bbox_to_anchor=(0.5, -0.12), 
    ncol=5, 
    frameon=False, 
    title="Mean LR Score",
    title_fontsize=11
)
ax.add_artist(legend2)

# 保存图片（bbox_inches="tight" 能够确保右侧超出绘图区的文本不被切掉）
plt.savefig("FigD_Total_MP_to_EC_Clean_Key_Pairs.pdf", dpi=300, bbox_inches="tight")
plt.show()
print("-> 时序具体配受体对气泡图（右侧大括号版）已保存为：FigD_Total_MP_to_EC_Clean_Key_Pairs.pdf")


In [ ]:
import liana as li
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# =======================================================
# ⚠️ 必须修改的地方 1：填写你 adata 里面的亚群列名
# 比如可能是 "seurat_clusters", "sub_celltype", "celltype_minor"
# =======================================================
subcluster_col = "celltype_for_liana" 

main_time_points = ["D14", "D33", "D61"]
adata_main = adata_cross[adata_cross.obs["time_point"].isin(main_time_points)].copy()
results_by_time_sub = {}

for tp in main_time_points:
    print(f"\nRunning Sub-cluster LIANA for {tp}...")
    
    adata_tp = adata_main[adata_main.obs["time_point"] == tp].copy()
    
    # 打印一下当前的亚群分布
    print(pd.crosstab(adata_tp.obs["time_point"], adata_tp.obs[subcluster_col]))

    # 按【亚群列】进行分组运算
    li.mt.rank_aggregate(
        adata_tp,
        groupby=subcluster_col,
        resource_name="consensus",
        expr_prop=0.1,
        min_cells=10,
        use_raw=False,
        verbose=True
    )

    res = adata_tp.uns["liana_res"].copy()
    res["time_point"] = tp
    results_by_time_sub[tp] = res



In [ ]:
# 检查原始数据中是否有 FCN+MP
print("检查原始数据中的细胞类型:")
print("="*50)

# 检查 source 列
sources = subcluster_liana_res['source'].unique()
targets = subcluster_liana_res['target'].unique()
all_cells = set(sources) | set(targets)

print(f"所有细胞类型 ({len(all_cells)} 个):")
for cell in sorted(all_cells):
    print(f"  - '{cell}'")

# 查找包含 FCN 的细胞
fcn_cells = [c for c in all_cells if 'FCN' in c or 'fcn' in c.lower()]
print(f"\n包含 'FCN' 的细胞: {fcn_cells}")

# 查找所有包含 MP 的细胞
mp_cells = [c for c in all_cells if 'MP' in c or 'mp' in c.lower()]
print(f"所有 MP 相关细胞: {mp_cells}")

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# 1. 尝试合并刚才跑完的时间点结果
try:
    subcluster_liana_res = pd.concat(results_by_time_sub.values(), ignore_index=True)
    print("✅ 成功加载了所有时间点的 LIANA 结果。")
except Exception as e:
    print("⚠️ 无法加载结果，请确保你前面的 LIANA 循环已经跑完，且 results_by_time_sub 存在。")

# 2. 打印看看你的结果表里到底有哪些列
print("LIANA 结果表的列名有：", subcluster_liana_res.columns.tolist())

# 3. 自动识别显著性指标列名 (绝对不会再报 KeyError)
if "cellphone_pvals" in subcluster_liana_res.columns:
    p_col = "cellphone_pvals"
elif "specificity_rank" in subcluster_liana_res.columns:
    p_col = "specificity_rank"
elif "aggregate_rank" in subcluster_liana_res.columns:
    p_col = "aggregate_rank"
else:
    # 如果上面的都没有，找第一个带有 pval 或 rank 的列
    possible_cols = [c for c in subcluster_liana_res.columns if 'pval' in c.lower() or 'rank' in c.lower()]
    p_col = possible_cols[0] if possible_cols else subcluster_liana_res.columns[-1]

print(f"--> 正在使用 [{p_col}] 列，过滤 P<=0.05 的显著通讯...")
sub_sig = subcluster_liana_res[subcluster_liana_res[p_col] <= 0.05].copy()

# =======================================================
# 🔥 定义你的细胞亚群（根据你的实际情况修改）
# =======================================================
# 巨噬细胞亚群
mp_subclusters = ['FCN1+MP', 'MP1', 'MARCO+MP']

# 自动识别内皮细胞亚群
all_cell_types = list(set(sub_sig['source'].unique()) | set(sub_sig['target'].unique()))
ec_subclusters = sorted([cell for cell in all_cell_types 
                         if 'EC' in cell or 'Endo' in cell or 'endothelial' in cell.lower() 
                         and cell not in mp_subclusters])

# 如果自动识别失败，手动指定
if not ec_subclusters:
    print("⚠️ 未检测到内皮细胞亚群，请手动指定")
    # 根据你的实际情况修改这里
    ec_subclusters = ['EC1', 'EC2', 'EC3']  # ← 改成你的实际内皮细胞名称
    # 或者从数据中自动提取所有非MP的细胞
    # ec_subclusters = [cell for cell in all_cell_types if cell not in mp_subclusters]

print(f"✅ 巨噬细胞亚群: {mp_subclusters}")
print(f"✅ 内皮细胞亚群: {ec_subclusters}")

# =======================================================
# 终极发表级美化版 (方块化 + 隐藏0 + 坐标文字上色)
# =======================================================
# ✅ 修复：将 sub_sig_filtered 改为 sub_sig
if sub_sig.shape[0] > 0:
    interaction_counts = sub_sig.groupby(['source', 'target']).size().reset_index(name='count')
    heatmap_data = interaction_counts.pivot(index='source', columns='target', values='count').fillna(0)

    # 排序：巨噬细胞在前，内皮细胞在后
    all_ordered_cells = mp_subclusters + ec_subclusters
    ordered_rows = [c for c in all_ordered_cells if c in heatmap_data.index]
    ordered_cols = [c for c in all_ordered_cells if c in heatmap_data.columns]
    
    # 如果ordered_rows或ordered_cols为空，使用所有可用的细胞
    if not ordered_rows:
        ordered_rows = list(heatmap_data.index)
        print("⚠️ 未找到匹配的源细胞，使用所有可用细胞")
    if not ordered_cols:
        ordered_cols = list(heatmap_data.columns)
        print("⚠️ 未找到匹配的目标细胞，使用所有可用细胞")
    
    heatmap_data = heatmap_data.loc[ordered_rows, ordered_cols]

    # ⭐ 核心美化 1：把 0 变成空白字符，减少视觉干扰
    annot_labels = heatmap_data.applymap(lambda x: str(int(x)) if x > 0 else "")

    # 设置画布大小
    plt.figure(figsize=(10, 9))
    
    # 绘制热图
    ax = sns.heatmap(
        heatmap_data, 
        cmap="Blues",                              
        annot=annot_labels,          
        fmt="",                      
        annot_kws={"size": 12, "weight": "bold"},  
        linewidths=1.5,              
        linecolor='white',
        square=True,                 
        cbar_kws={"shrink": 0.75}    
    )

    plt.title("Significant Interactions: Endothelial vs Macrophages", fontsize=16, fontweight='bold', pad=20)
    plt.xlabel("Target Cells (Receiver)", fontsize=14, labelpad=15)
    plt.ylabel("Source Cells (Sender)", fontsize=14, labelpad=15)
    
    # ⭐ 核心美化 3：为坐标轴标签直接上色
    mp_color = '#C41E3A' # 砖红色
    ec_color = '#1E3A8A' # 蓝绿色
    
    # 修改X轴文字颜色和字体
    for tick_label in ax.get_xticklabels():
        tick_text = tick_label.get_text()
        tick_label.set_color(mp_color if tick_text in mp_subclusters else ec_color)
        tick_label.set_fontsize(12)
        tick_label.set_weight("bold")
    plt.xticks(rotation=45, ha='right')

    # 修改Y轴文字颜色和字体
    for tick_label in ax.get_yticklabels():
        tick_text = tick_label.get_text()
        tick_label.set_color(mp_color if tick_text in mp_subclusters else ec_color)
        tick_label.set_fontsize(12)
        tick_label.set_weight("bold")
    plt.yticks(rotation=0)

    # ⭐ 核心美化 4：将十字分割线换成精致的黑色虚线
    mp_count_in_rows = len([c for c in mp_subclusters if c in heatmap_data.index])
    mp_count_in_cols = len([c for c in mp_subclusters if c in heatmap_data.columns])
    
    if 0 < mp_count_in_rows < len(ordered_rows):
        ax.axhline(mp_count_in_rows, color='black', lw=1.5, linestyle='--', alpha=0.7) 
    if 0 < mp_count_in_cols < len(ordered_cols):
        ax.axvline(mp_count_in_cols, color='black', lw=1.5, linestyle='--', alpha=0.7) 

    plt.tight_layout()
    plt.savefig("Subcluster_Interaction_Heatmap_UltraBeautiful.pdf", dpi=300, bbox_inches="tight")
    plt.show()
else:
    print("没有找到符合条件的显著通讯，无法绘制热图。")

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# =========================================================
# 1. 确保细胞亚群的定义
# =========================================================
mp_subclusters = ["FCN1+MP"]
ec_subclusters = [
    "Arterial EC", 
    "ECM-remodeling Arterial EC", 
    "Venous EC", 
    "Capillary EC_1", 
    "Capillary EC_2", 
    "Glomerular EC"
]

# 确保时间点有序
sub_sig["time_point"] = pd.Categorical(
    sub_sig["time_point"], 
    categories=["D14", "D33", "D61"], 
    ordered=True
)

# =========================================================
# 2. 提取并计算两个方向的时序通讯总数
# =========================================================
# 基础统计：每个时间点、Sender、Receiver 的显著通讯对数
counts_df = sub_sig.groupby(["time_point", "source", "target"], observed=False).size().reset_index(name="count")

# --- 方向一：MP -> EC (EC 作为接收端 Target) ---
mp_to_ec_data = counts_df[
    counts_df["source"].isin(mp_subclusters) & counts_df["target"].isin(ec_subclusters)
].copy()
# 按【时间点】和【EC亚群】汇总所有 MP 发送给它的信号总数
mp_to_ec_trend = mp_to_ec_data.groupby(["time_point", "target"], observed=False)["count"].sum().reset_index()
mp_to_ec_trend.rename(columns={"target": "EC_Subcluster"}, inplace=True)

# --- 方向二：EC -> MP (EC 作为发送端 Source) ---
ec_to_mp_data = counts_df[
    counts_df["source"].isin(ec_subclusters) & counts_df["target"].isin(mp_subclusters)
].copy()
# 按【时间点】和【EC亚群】汇总它发送给所有 MP 的信号总数
ec_to_mp_trend = ec_to_mp_data.groupby(["time_point", "source"], observed=False)["count"].sum().reset_index()
ec_to_mp_trend.rename(columns={"source": "EC_Subcluster"}, inplace=True)

# =========================================================
# 3. 开始绘制折线图 (1行2列并排展示)
# =========================================================
fig, axes = plt.subplots(1, 2, figsize=(13, 5.5), sharey=True)
sns.set_theme(style="ticks")

# 定义每个 EC 亚群固定的颜色和线型，确保左右两张图一一对应，方便对比
ec_colors = {
    "Arterial EC": "#E74C3C",           # 红色
    "ECM-remodeling Arterial EC": "#F39C12", # 橙色
    "Venous EC": "#3498DB",             # 蓝色
    "Capillary EC_1": "#2ECC71",        # 绿色
    "Capillary EC_2": "#27AE60",        # 深绿
    "Glomerular EC": "#9B59B6"          # 紫色
}

markers = {
    "Arterial EC": "o",
    "ECM-remodeling Arterial EC": "s",
    "Venous EC": "^",
    "Capillary EC_1": "v",
    "Capillary EC_2": "D",
    "Glomerular EC": "p"
}

# ---------------------------------------------------------
# 子图 1：MP -> EC (EC 接收的通讯数)
# ---------------------------------------------------------
ax1 = axes[0]
for ec in ec_subclusters:
    sub_data = mp_to_ec_trend[mp_to_ec_trend["EC_Subcluster"] == ec]
    ax1.plot(
        sub_data["time_point"].astype(str), 
        sub_data["count"], 
        marker=markers[ec], 
        color=ec_colors[ec], 
        linewidth=2.5, 
        markersize=8, 
        label=ec
    )
ax1.set_title("Incoming Signals (FCN1+MP $\\rightarrow$ EC Subclusters)", fontsize=13, fontweight='bold', pad=15)
ax1.set_ylabel("Number of Significant Interactions", fontsize=11, fontweight='bold')
ax1.set_xlabel("Time Point", fontsize=11, fontweight='bold')
ax1.grid(axis='both', linestyle=':', alpha=0.5)

# ---------------------------------------------------------
# 子图 2：EC -> MP (EC 发送的通讯数)
# ---------------------------------------------------------
ax2 = axes[1]
for ec in ec_subclusters:
    sub_data = ec_to_mp_trend[ec_to_mp_trend["EC_Subcluster"] == ec]
    ax2.plot(
        sub_data["time_point"].astype(str), 
        sub_data["count"], 
        marker=markers[ec], 
        color=ec_colors[ec], 
        linewidth=2.5, 
        markersize=8, 
        label=ec
    )
ax2.set_title("Outgoing Signals (EC Subclusters $\\rightarrow$ MP)", fontsize=13, fontweight='bold', pad=15)
ax2.set_xlabel("Time Point", fontsize=11, fontweight='bold')
ax2.grid(axis='both', linestyle=':', alpha=0.5)

# =========================================================
# 4. 细节调整与美化
# =========================================================
# 统一移除上方和右侧的边框线
sns.despine(fig, top=True, right=True)

# 将图例合并并统一放置在图表的右侧
handles, labels = ax1.get_legend_handles_labels()
fig.legend(
    handles, 
    labels, 
    loc="center right", 
    bbox_to_anchor=(1.18, 0.5), # 调整这里的坐标防止图例与右侧图表重叠
    frameon=False, 
    title="EC Subclusters", 
    title_fontsize=12,
    fontsize=11
)

plt.suptitle("Temporal Dynamics of Interactions Across All EC Subclusters", 
             fontsize=15, fontweight='bold', y=1.05)

# 留出右侧空间给 legend，避免遮挡
plt.tight_layout()
plt.subplots_adjust(right=0.85)

plt.savefig("All_EC_Temporal_Linecharts.pdf", dpi=300, bbox_inches="tight")
plt.show()
print("-> 所有内皮细胞亚群的时序折线图已保存为：All_EC_Temporal_Linecharts.pdf")


In [ ]:
import pandas as pd

# =========================================================
# 1. 定义发送端和接收端
# =========================================================
fcn1_mp = ["FCN1+MP"]
ec_subclusters = [
    "Arterial EC", "ECM-remodeling Arterial EC", "Venous EC", 
    "Capillary EC_1", "Capillary EC_2", "Glomerular EC"
]

# 提取 FCN1+MP -> 所有 EC 亚群的数据
df_check = sub_sig[
    (sub_sig["source"].isin(fcn1_mp)) & 
    (sub_sig["target"].isin(ec_subclusters))
].copy()

# =========================================================
# 2. 拼接配体和受体，生成唯一的交互名称
# =========================================================
# 把 ligand 和 receptor 拼起来，中间加个下划线 "_"
df_check["interacting_pair"] = df_check["ligand_complex"].astype(str) + "_" + df_check["receptor_complex"].astype(str)

# 提取并打印唯一交互对
unique_pairs = df_check["interacting_pair"].dropna().unique()

print(f"=== FCN1+MP -> ECs 方向共发现 {len(unique_pairs)} 种不同的受体-配体对 ===")
# 打印前 50 个看看（防止太多刷屏）
print(unique_pairs[:50]) 
print("====================================================\n")

# =========================================================
# 3. 制作多维度的汇总表，方便查阅
# =========================================================
# 方法 A：统计每个受体配体对在不同时间点，成功通讯的“频次”（命中几个EC亚群）
summary_counts = pd.crosstab(df_check["interacting_pair"], df_check["time_point"])
summary_counts.columns.name = "Time Point"
summary_counts.index.name = "Interacting Pair"

print("--- 每个受体-配体对在各时间点出现的频次预览 ---")
print(summary_counts.head(10)) 

# 保存频次统计表
summary_counts.to_csv("Check_FCN1MP_to_EC_Pair_Counts.csv")
print("\n-> 频次汇总表已保存为：Check_FCN1MP_to_EC_Pair_Counts.csv")

# 方法 B：保存明细表（提取您数据中重要的列，包含 lrscore 评分）
detail_cols = [
    "time_point", "source", "target", 
    "ligand_complex", "receptor_complex", "interacting_pair", 
    "lr_means", "lrscore", "cellphone_pvals"
]
# 仅保留存在的列，防止报错
existing_cols = [col for col in detail_cols if col in df_check.columns]
df_detail = df_check[existing_cols].sort_values(by=["time_point", "interacting_pair"])

df_detail.to_csv("Check_FCN1MP_to_EC_Detail.csv", index=False)
print("-> 完整交互明细表已保存为：Check_FCN1MP_to_EC_Detail.csv")


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# =========================================================
# 1. 提取 FCN1+MP -> EC 的数据，并打上宏观家族标签
# =========================================================
fcn1_mp = ["FCN1+MP"]

# 筛选靶细胞为 EC 结尾的细胞（兼容您所有的 EC 亚群）
df_fcn1_to_ec = sub_sig[
    (sub_sig["source"].isin(fcn1_mp)) & 
    (sub_sig["target"].str.contains("EC", na=False))
].copy()

# 宏观家族分类函数
def categorize_pathway(ligand):
    p = str(ligand).upper()
    if any(x in p for x in ["CXCL", "CCL", "IL", "TNF", "IFNG", "LTA", "CNTF"]): return "Chemokine / Cytokine"
    elif "COMPLEMENT" in p or "C3" in p or "C5" in p: return "Complement System"
    elif any(x in p for x in ["VEGF", "FGF", "PDGF", "ANGPT", "EGF", "IGF", "HGF", "JAG", "EDN", "EPGN"]): return "Growth Factor / Angiogenesis"
    elif any(x in p for x in ["SEMA", "EPH", "EFN"]): return "Guidance / Ephrin Signaling"
    elif any(x in p for x in ["TGF", "BMP", "GDF", "INHBA", "GREM"]): return "TGF-b / BMP Signaling"
    elif any(x in p for x in ["COL", "LAM", "FN1", "CD99", "PECAM", "ICAM", "VCAM", "MMP", "CDH", "DCN"]): return "Adhesion / ECM / Matrix"
    else: return "Metabolic / Calcium / Others"

df_fcn1_to_ec['Signaling Family'] = df_fcn1_to_ec['ligand_complex'].apply(categorize_pathway)

# =========================================================
# 2. 准备统一的坐标轴和色带标准
# =========================================================
ordered_times = ["D14", "D33", "D61"]
ordered_families = [
    "Adhesion / ECM / Matrix", "Chemokine / Cytokine", "Complement System",
    "Growth Factor / Angiogenesis", "Guidance / Ephrin Signaling",
    "TGF-b / BMP Signaling", "Metabolic / Calcium / Others"
]

# 找到实际存在的 EC 亚群
targets = sorted(df_fcn1_to_ec['target'].unique())
n_targets = len(targets)

# 计算全局的均值上下限，用于统一所有子图的颜色深浅
global_mean = df_fcn1_to_ec.groupby(['target', 'time_point', 'Signaling Family'], observed=False)['lrscore'].mean()
vmax_global = global_mean.max()
vmin_global = 0  # 您可以设为 global_mean.min() 或直接 0

# =========================================================
# 3. 动态计算画板大小并循环画图
# =========================================================
cols = 3  # 每行放3个图
rows = (n_targets + cols - 1) // cols
fig, axes = plt.subplots(rows, cols, figsize=(5 * cols, 5 * rows))
axes = axes.flatten()  # 展平以便于循环

for i, target_ec in enumerate(targets):
    ax = axes[i]
    
    # 提取当前 EC 的数据
    df_sub = df_fcn1_to_ec[df_fcn1_to_ec['target'] == target_ec]
    
    # 计算分组均值
    heat_data = df_sub.groupby(['time_point', 'Signaling Family'], observed=False)['lrscore'].mean().reset_index()
    pivot_df = heat_data.pivot(index='Signaling Family', columns='time_point', values='lrscore')
    
    # 补全缺失的时间点和通路，确保所有子图的格子尺寸一模一样
    for t in ordered_times:
        if t not in pivot_df.columns: pivot_df[t] = np.nan
    pivot_df = pivot_df[ordered_times]
    pivot_df = pivot_df.reindex(ordered_families)
    
    # 画子图热图
    sns.heatmap(
        pivot_df, 
        cmap="Blues", 
        annot=True, 
        fmt=".2f", 
        linewidths=1.2, 
        linecolor='white',
        vmin=vmin_global,    # 核心：统一最小颜色值
        vmax=vmax_global,    # 核心：统一最大颜色值
        cbar_kws={'shrink': 0.8} if i % cols == cols - 1 else None, # 尽量只在右侧图保留Colorbar
        cbar=False if i % cols != cols - 1 else True, # 避免每个图都有长长的尾巴
        ax=ax
    )
    
    # 样式美化
    ax.set_facecolor('#FFFFFF') # 无数据处为浅灰色
    for _, spine in ax.spines.items():
        spine.set_visible(True)
        spine.set_color('black')
        spine.set_linewidth(1.2)
        
    ax.set_title(f"Target: {target_ec}", fontsize=13, fontweight='bold', pad=10)
    ax.set_xlabel("Time Point", fontweight='bold')
    
    # 只有最左边那一列显示 Y 轴的 Family 名字，让整图更清爽
    if i % cols == 0:
        ax.set_ylabel("Signaling Family", fontweight='bold')
        ax.tick_params(axis='y', rotation=0)
    else:
        ax.set_ylabel("")
        ax.set_yticks([]) 

# 隐藏多余的空白子图 (如果 EC 数量不是 3 的倍数)
for j in range(i + 1, len(axes)):
    fig.delaxes(axes[j])

# 全局标题
fig.suptitle("FCN1+MP Macro-Pathway Signals to Distinct EC Subpopulations", 
             fontsize=18, fontweight='bold', y=1.02)

plt.tight_layout()
plt.savefig("MacroPathway_FCN1MP_to_EachEC_Grid.pdf", dpi=300, bbox_inches="tight")
plt.show()

print("-> 已生成分面板对比热图，并保存为：MacroPathway_FCN1MP_to_EachEC_Grid.pdf")


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches

# =========================================================
# 0. 全局字体与样式设置
# =========================================================
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['font.sans-serif'] = ['Arial', 'Helvetica', 'DejaVu Sans']

# =========================================================
# 1. 提取数据
# =========================================================
df_core = df_fcn1_to_ec.copy()
df_core["LR_Pair"] = df_core["ligand_complex"].astype(str) + " → " + df_core["receptor_complex"].astype(str)

top_lrs = df_core.groupby(["Signaling Family", "LR_Pair"], observed=False)['lrscore'].max().reset_index()
top_lrs = top_lrs.sort_values(['Signaling Family', 'lrscore'], ascending=[True, False])
keep_lrs = top_lrs.groupby("Signaling Family").head(4)["LR_Pair"].tolist()

plot_df = df_core[df_core["LR_Pair"].isin(keep_lrs)].copy()

# =========================================================
# 2. 生成 Y 轴坐标
# =========================================================
ordered_families = [
    "TGF-b / BMP Signaling", "Chemokine / Cytokine", "Guidance / Ephrin Signaling", 
    "Growth Factor / Angiogenesis", "Adhesion / ECM / Matrix", "Complement System", 
    "Metabolic / Calcium / Others"
]

plot_df['Family_Cat'] = pd.Categorical(plot_df['Signaling Family'], categories=ordered_families, ordered=True)
plot_df = plot_df.sort_values(by=["Family_Cat", "LR_Pair"], ascending=[False, False]) 

y_labels = plot_df["LR_Pair"].unique()
y_mapping = {label: i for i, label in enumerate(y_labels)}
plot_df["Y_Pos"] = plot_df["LR_Pair"].map(y_mapping)

# =========================================================
# 3. 生成 X 轴坐标
# =========================================================
targets = sorted(plot_df["target"].unique())
times = ["D14", "D33", "D61"]

x_labels = []
x_mapping = {}
idx = 0
for tgt in targets:
    for t in times:
        x_labels.append(f"{tgt} | {t}")
        x_mapping[(tgt, t)] = idx
        idx += 1

plot_df["X_Pos"] = plot_df.apply(lambda row: x_mapping.get((row["target"], row["time_point"]), -1), axis=1)
plot_df = plot_df[plot_df["X_Pos"] != -1]

if 'cellphone_pvals' not in plot_df.columns:
    plot_df['cellphone_pvals'] = 0.01 
plot_df["-log10(P)"] = -np.log10(plot_df["cellphone_pvals"].astype(float) + 1e-4)

SIZE_SCALE = 50
SIZE_BASE = 30
plot_df["bubble_size"] = plot_df["-log10(P)"] * SIZE_SCALE + SIZE_BASE

# =========================================================
# 4. 绘制超精美气泡图
# =========================================================
# 保持画布尺寸，但稍后我们将控制右侧留白
fig, ax = plt.subplots(figsize=(15, 11))  

# 调整主图区域，把右侧 28% 的空间全部留给文本和图例
fig.subplots_adjust(right=0.72)

shade = False
for family in plot_df["Family_Cat"].dropna().unique():
    sub = plot_df[plot_df["Family_Cat"] == family]
    if sub.empty: continue
        
    ymin, ymax = sub["Y_Pos"].min() - 0.5, sub["Y_Pos"].max() + 0.5
    if shade:
        ax.axhspan(ymin, ymax, facecolor='#F4F6F9', edgecolor='none', zorder=0, alpha=0.8)
    shade = not shade
    
    # 家族名字：缩短距离，紧贴图表右侧，提升关联感
    ax.text(x=len(x_labels) + 0.2, y=(ymin+ymax)/2, s=family, 
            va='center', ha='left', fontweight='bold', color='#2C3E50', fontsize=11, clip_on=False)
    ax.axhline(ymax, color='#D5D8DC', lw=0.6, zorder=1)
ax.axhline(-0.5, color='#D5D8DC', lw=0.6, zorder=1)

# 气泡散点
scatter = ax.scatter(
    plot_df["X_Pos"], plot_df["Y_Pos"],
    s=plot_df["bubble_size"], 
    c=plot_df["lrscore"],
    cmap="Blues",                      
    edgecolors="#0B3C68",      
    linewidths=0.8,         
    alpha=0.95,
    zorder=3
)

# 纵向分割线与顶部分组线
for i in range(1, len(targets)):
    ax.axvline(i * 3 - 0.5, color='#BDC3C7', lw=1, linestyle='--', alpha=0.5, zorder=1)

y_top_line = len(y_labels) + 0.1 
y_top_text = len(y_labels) + 0.4 

for i, tgt in enumerate(targets):
    start_x = i * 3
    end_x = start_x + 2
    center_x = (start_x + end_x) / 2
    
    ax.plot([start_x, end_x], [y_top_line, y_top_line], color='#2C3E50', lw=1.5, zorder=4)
    ax.plot([start_x, start_x], [y_top_line-0.1, y_top_line], color='#2C3E50', lw=1.5, zorder=4)
    ax.plot([end_x, end_x], [y_top_line-0.1, y_top_line], color='#2C3E50', lw=1.5, zorder=4)
    
    ax.text(x=center_x, y=y_top_text, s=tgt,
            ha='center', va='bottom', fontsize=11, fontweight='bold', color='#1A5276')

# 绘制黑色细边框
for tgt in targets:
    tgt_df = plot_df[plot_df["target"] == tgt]
    if tgt_df.empty: continue
    max_idx = tgt_df['lrscore'].idxmax()
    max_row = tgt_df.loc[max_idx]
    max_x, max_y = max_row["X_Pos"], max_row["Y_Pos"]
    
    highlight_box = patches.FancyBboxPatch(
        (max_x - 0.4, max_y - 0.4), 0.8, 0.8, 
        boxstyle="round,pad=0.02,rounding_size=0.05", 
        linewidth=1.5, edgecolor='#17202A', facecolor='none', zorder=5                      
    )
    ax.add_patch(highlight_box)

# =========================================================
# 5. 坐标轴美化
# =========================================================
ax.set_yticks(range(len(y_labels)))
ax.set_yticklabels([f"${label.split(' → ')[0]}$ → ${label.split(' → ')[1]}$" 
                    if '_' not in label else label for label in y_labels], 
                   fontstyle='italic', fontsize=10)

ax.set_xticks(range(len(x_labels)))
clean_x_labels = [label.split(" | ")[1] for label in x_labels]
ax.set_xticklabels(clean_x_labels, rotation=45, ha='right', fontsize=10)

ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.spines['bottom'].set_visible(False)
ax.spines['left'].set_linewidth(1.2)

# 收紧 X 轴，不再让右侧无限拉长
ax.set_xlim(-0.5, len(x_labels) - 0.5)
ax.set_ylim(-1, len(y_labels) + 1.5) 

# =========================================================
# 6. 右侧图例终极排版 (Legends & Colorbar)
# =========================================================
# 6.1 P-value 图例：固定在右侧上方
p_vals = [0.05, 0.01, 0.001]
log_ps = [-np.log10(p) for p in p_vals]
sizes = [lp * SIZE_SCALE + SIZE_BASE for lp in log_ps]

legend_markers = [plt.scatter([], [], s=s, c='#85C1E9', edgecolors='#0B3C68', lw=0.8) for s in sizes]
# 调整 bbox_to_anchor 将其完美放置在文字右侧的独立区域
leg = ax.legend(legend_markers, ['0.05', '0.01', '0.001'], title='P-value', 
                loc='upper left', bbox_to_anchor=(1.35, 0.95), 
                frameon=False, labelspacing=1.3, title_fontsize=11)
leg._legend_box.align = "left"

# 6.2 优雅细长的 Colorbar：固定在 P-value 图例正下方
# 使用绝对坐标 [left, bottom, width, height] 添加一个细长的坐标轴给色条
cbar_ax = fig.add_axes([0.895, 0.35, 0.012, 0.35]) 
cbar = plt.colorbar(scatter, cax=cbar_ax)
cbar.set_label('Communication Prob. (lrscore)', rotation=270, labelpad=18, fontweight='bold', fontsize=11)
cbar.outline.set_linewidth(0.8)
cbar.ax.tick_params(labelsize=10)

# =========================================================
# 7. 标题
# =========================================================
plt.text(0.4, 1.08, "Spatiotemporal Dynamics of Cell Signaling", 
         fontsize=16, fontweight='bold', ha='center', transform=ax.transAxes)
plt.text(0.4, 1.04, "(Black boxes highlight the strongest signal per target cluster)", 
         fontsize=12, style='italic', color='#566573', ha='center', transform=ax.transAxes)

plt.savefig("AllFamilies_PerfectLayout_Styled.pdf", dpi=300, bbox_inches="tight")
plt.show()

print("-> 右侧排版已全面优化！")


In [ ]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
import matplotlib.colors as mcolors

# =========================================================
# 1. 数据准备
# =========================================================
# 复制您的原始 DataFrame
df_sankey = df_fcn1_to_ec.copy()

# 筛选 lrscore 最显著的 Top 25 信号
df_top25 = df_sankey.nlargest(25, 'lrscore').copy()

# 格式化基因名称：斜体 + 优雅的箭头
df_top25["LR_Pair"] = "<i>" + df_top25["ligand_complex"].astype(str) + "</i> → <i>" + df_top25["receptor_complex"].astype(str) + "</i>"

# 提取唯一的左侧（LR Pair）与右侧（Target EC）节点列表
sources = df_top25['LR_Pair'].unique().tolist()
targets = df_top25['target'].unique().tolist()
all_nodes = sources + targets

# 建立节点索引映射
node_mapping = {node: i for i, node in enumerate(all_nodes)}

# =========================================================
# 2. 配色方案（使用学术期刊爱用的莫兰迪/Nature色系）
# =========================================================
target_colors = {
    "ECM-remodeling Arterial EC": "#E64B35", # 活力红
    "Arterial EC": "#4DBBD5",           # 明亮蓝
    "Capillary EC_1": "#00A087",        # 翡翠绿
    "Capillary EC_2": "#3C5488",        # 深邃蓝
    "Glomerular EC": "#F39B7F",         # 柔和橘
    "Venous EC": "#8491B4"              # 灰蓝色
}

node_colors = []
for node in all_nodes:
    if node in targets:
        node_colors.append(target_colors.get(node, "#808080"))
    else:
        # 左侧分子颜色追随其主要靶向（lrscore最大）的细胞颜色
        main_target = df_top25[df_top25['LR_Pair'] == node].nlargest(1, 'lrscore')['target'].values[0]
        node_colors.append(target_colors.get(main_target, "#808080"))

# 十六进制颜色转 rgba，用于控制连线丝带的透明度
def hex_to_rgba(hex_color, alpha=0.35):
    rgb = mcolors.hex2color(hex_color)
    return f"rgba({int(rgb[0]*255)}, {int(rgb[1]*255)}, {int(rgb[2]*255)}, {alpha})"

# =========================================================
# 3. 构建 Sankey 连线
# =========================================================
link_source = []
link_target = []
link_value = []
link_color = []

for _, row in df_top25.iterrows():
    s_idx = node_mapping[row['LR_Pair']]
    t_idx = node_mapping[row['target']]
    
    link_source.append(s_idx)
    link_target.append(t_idx)
    link_value.append(row['lrscore'])
    # 丝带半透明化，防止重叠时看起来杂乱
    link_color.append(hex_to_rgba(node_colors[t_idx], alpha=0.3))

# =========================================================
# 4. 构建图表
# =========================================================
fig = go.Figure(data=[go.Sankey(
    arrangement='snap', # 算法自动减少线条交叉
    node=dict(
        pad=18,         # 节点垂直间隔
        thickness=20,   # 节点宽度
        line=dict(color="#2C3E50", width=0.5), # 节点加极细边框
        label=all_nodes,
        color=node_colors
    ),
    link=dict(
        source=link_source,
        target=link_target,
        value=link_value,
        color=link_color
    )
)])

# =========================================================
# 5. 全局精美排版设置
# =========================================================
fig.update_layout(
    title_text="<b>Top 25 Most Significant Signals:</b> FCN1+MP to EC Subpopulations",
    title_font_size=18,
    title_x=0.05, # 标题微靠左偏
    font=dict(
        family="Arial, Helvetica, sans-serif",
        size=12,
        color="#2C3E50"
    ),
    plot_bgcolor='white',
    paper_bgcolor='white',
    width=950,
    height=700,
    margin=dict(l=40, r=40, t=80, b=40)
)

# =========================================================
# 6. 免配置输出（直接生成网页并打开）
# =========================================================
# 1. 导出为交互式网页 HTML 
fig.write_html("Sankey_Premium_Macrophage_to_EC.html")

# 2. 在 Notebook 中展示
fig.show()

print("-> 成功！请在代码运行目录下寻找并双击打开 'Sankey_Premium_Macrophage_to_EC.html' 文件。")


In [ ]:
# 1. 自动获取最高的受体配体对
top_1_pair = df_fcn1_to_ec.nlargest(1, 'lrscore').iloc[0]
ligand_name = top_1_pair['ligand_complex']
receptor_name = top_1_pair['receptor_complex']

# 拆分可能存在的复合物基因（如 "ITGB1_ITGA5" -> ["ITGB1", "ITGA5"]）
genes_to_plot = []
for g in [ligand_name, receptor_name]:
    genes_to_plot.extend(g.split('_'))
genes_to_plot = list(set(genes_to_plot))  # 去重

# 2. 确定您的细胞类型列名（请将 'cell_type' 修改为您 adata.obs 中实际的列名，如 'celltype' 或 'cluster'）
cell_type_key = 'celltype_for_liana' 

# 获取所有的细胞类型列表并排序
all_cell_types = sorted(adata_cross.obs[cell_type_key].unique().tolist())
print(f"检测到以下 {len(all_cell_types)} 种细胞类型进行绘制：\n", all_cell_types)
print(f"准备绘制基因: {genes_to_plot}")


In [ ]:
import scanpy as sc
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

# 设置全局绘图风格为干净无干扰
sns.set_theme(style="ticks")
plt.rcParams['font.sans-serif'] = ['Arial', 'Helvetica'] # 使用学术常用字体
plt.rcParams['axes.unicode_minus'] = False

# 1. 提取基因在各细胞类型中的平均表达量
genes = ['VEGFD', 'FLT1', 'ICAM1', 'CAV1', 'NAMPT', 'INSR', 'QDPR', 'DYSF']
ordered_cell_types = [
    'FCN1+MP', 'MARCO+MP', 'MP1', 
    'ECM-remodeling Arterial EC', 'Arterial EC', 
    'Capillary EC_1', 'Capillary EC_2', 
    'Glomerular EC', 'Venous EC'
]

# 计算平均表达量并归一化到 0-1 (Min-Max)
mean_expr = pd.DataFrame(index=ordered_cell_types)
for gene in genes:
    expr = adata_cross[:, gene].X.toarray().flatten() if hasattr(adata_cross[:, gene].X, "toarray") else adata_cross[:, gene].X.flatten()
    df_tmp = pd.DataFrame({'Expr': expr, 'Celltype': adata_cross.obs['celltype_for_liana']})
    cell_means = df_tmp.groupby('Celltype')['Expr'].mean().reindex(ordered_cell_types).fillna(0)
    mean_expr[gene] = (cell_means - cell_means.min()) / (cell_means.max() - cell_means.min() + 1e-8)

# 2. 创建子图 (调整比例让它看起来宽敞优雅)
fig, axes = plt.subplots(1, 4, figsize=(18, 4.5), sharey=True)
pairs = [
    ("VEGFD", "FLT1", "Angiogenesis"),
    ("ICAM1", "CAV1", "Adhesion"),
    ("NAMPT", "INSR", "Metabolic"),
    ("QDPR", "DYSF", "Metabolic / Calcium")
]

# 高级莫兰迪学术配色
colors = {'Ligand': '#D65F5F', 'Receptor': '#4878CF'} 

for i, (ligand, receptor, title) in enumerate(pairs):
    ax = axes[i]
    
    # 绘制折线（柔化线段宽度，加粗描点边缘）
    ax.plot(ordered_cell_types, mean_expr[ligand], marker='o', color=colors['Ligand'], 
            label=f"Ligand: {ligand}", lw=2, markersize=8, markeredgecolor='white', markeredgewidth=1)
    ax.plot(ordered_cell_types, mean_expr[receptor], marker='s', color=colors['Receptor'], 
            label=f"Receptor: {receptor}", lw=2, markersize=8, markeredgecolor='white', markeredgewidth=1)
    
    # 标题设计 (增大间距与字号)
    ax.set_title(f"{title}\n({ligand} $\\rightarrow$ {receptor})", fontsize=13, fontweight='bold', pad=15, color='#222222')
    
    # X轴标签优化：旋转45度，向右对齐，避免重叠
    ax.set_xticks(range(len(ordered_cell_types)))
    ax.set_xticklabels(ordered_cell_types, rotation=45, ha='right', fontsize=11, color='#333333')
    
    # 去除垂直背景线，只保留柔和的水平网格引导线
    ax.grid(axis='y', linestyle='--', color='#E0E0E0', alpha=0.8, zorder=0)
    ax.set_axisbelow(True) # 保证网格线在折线下方
    
    # 坐标轴美化 (改成深灰)
    ax.spines['left'].set_color('#555555')
    ax.spines['bottom'].set_color('#555555')
    ax.tick_params(colors='#555555', labelsize=10.5)
    
    if i == 0:
        ax.set_ylabel("Normalized Mean Expression", fontsize=12, fontweight='bold', color='#222222')
    
    # 图例美化 (移至左上角，彻底去掉框线，背景透明，防止遮挡最高点)
    ax.legend(loc='upper left', frameon=False, fontsize=9.5, handletextpad=0.5)

# 移除上边框和右边框
sns.despine()

plt.tight_layout()
# 导出为 PDF
plt.savefig("LR_Pairs_Line_Profile_Polished.pdf", dpi=300, bbox_inches='tight')
plt.show()


In [ ]:
# ============================================================
# 修复版：评估 1-to-1 与 1-to-many / many-to-many 的分布及受配体覆盖
# ============================================================
import pandas as pd
import liana as li

# 1. 统计各类同源基因的比例分布
orth_types = orth['orthology_type'].value_counts()
orth_types_pct = orth['orthology_type'].value_counts(normalize=True) * 100
summary_df = pd.DataFrame({
    'Count': orth_types, 
    'Percentage (%)': orth_types_pct.round(2)
})
print("=== 1. 猪与人的同源类型分布 ===")
print(summary_df)

# 2. 提取被 1-to-1 策略过滤掉的非 1-to-1 基因
excluded_orth = orth[orth['orthology_type'] != 'ortholog_one2one'].dropna(subset=['pig_gene', 'human_gene'])
excluded_human_genes = set(excluded_orth['human_gene'].astype(str).unique())
excluded_pig_genes = set(excluded_orth['pig_gene'].astype(str).unique())

print(f"\n排除的非 1-to-1 猪基因数: {len(excluded_pig_genes)}")
print(f"排除的非 1-to-1 人直系同源基因数: {len(excluded_human_genes)}")

# 3. 检查你关注的核心受配体清单 (key_lr_genes) 有没有被误过滤
retained_1to1_human_genes = set(pig_to_human.values())
excluded_key_genes = [g for g in key_lr_genes if g in excluded_human_genes and g not in retained_1to1_human_genes]
print(f"\n核心受配体清单中被排除的基因: {excluded_key_genes} (如果为空列表 [] 说明核心靶点 100% 都在)")

# 4. 获取 LIANA 官方 consensus 数据库中的全部受配体基因
try:
    # LIANA 标准接口是 select_resource
    lr_db = li.rs.select_resource('consensus')
except AttributeError:
    try:
        lr_db = li.resource.select_resource('consensus')
    except Exception:
        # 如果当前环境受限，直接从你刚刚跑完的 LIANA 结果里提取
        print("从当前结果提取配受体...")
        lr_db = ec_mp_cross_res[['ligand.complex', 'receptor.complex']].rename(
            columns={'ligand.complex': 'ligand', 'receptor.complex': 'receptor'}
        )

# 提取受配体库中涉及的所有人类基因 symbol
lr_genes_in_db = set(lr_db['ligand'].dropna().tolist() + lr_db['receptor'].dropna().tolist())

# 统计受配体基因在 1-to-1 和非 1-to-1 中的保留情况
lr_in_one2one = len(lr_genes_in_db.intersection(retained_1to1_human_genes))
lr_in_excluded_only = len(lr_genes_in_db.intersection(excluded_human_genes) - retained_1to1_human_genes)

print(f"\n=== 2. LIANA 受配体库覆盖情况 ===")
print(f"成功保留在 1-to-1 映射中的受配体基因数: {lr_in_one2one}")
print(f"属于非 1-to-1 且被排除的受配体基因数: {lr_in_excluded_only}")
retention_rate = lr_in_one2one / (lr_in_one2one + lr_in_excluded_only) * 100
print(f"1-to-1 策略对受配体库的保留率: {retention_rate:.2f}%")

In [ ]:
# 打印当前结果中的关键列名
print(ec_mp_cross_res.columns.tolist())

# 看看是否有自带的 fdr/padj 列，或者查看有多少互作满足 P < 0.05
# 通常 LIANA rank_aggregate 会生成:
# 'magnitude_rank', 'specificity_rank' 或 'aggregate_rank'

In [ ]:
import pandas as pd
from statsmodels.stats.multitest import multipletests

# 1. 针对每一个时间点（D14, D33, D61）分别做 Benjamini-Hochberg (BH) FDR 校正
ec_mp_cross_res['cellphone_padj'] = 1.0  # 初始化

for tp in ec_mp_cross_res['time_point'].unique():
    mask = ec_mp_cross_res['time_point'] == tp
    pvals = ec_mp_cross_res.loc[mask, 'cellphone_pvals']
    
    # 执行 BH-FDR 校正
    _, padj, _, _ = multipletests(pvals, method='fdr_bh')
    ec_mp_cross_res.loc[mask, 'cellphone_padj'] = padj

# 2. 统计校正前后的显著互作数量
raw_sig = (ec_mp_cross_res['cellphone_pvals'] < 0.05).sum()
fdr_sig = (ec_mp_cross_res['cellphone_padj'] < 0.05).sum()
total_tested = len(ec_mp_cross_res)

print(f"=== 多重假设检验校正 (BH-FDR) 统计 ===")
print(f"测试的总互作数: {total_tested}")
print(f"名义显著数 (Raw P < 0.05): {raw_sig} ({raw_sig/total_tested*100:.1f}%)")
print(f"FDR 校正后显著数 (Adjusted P < 0.05): {fdr_sig} ({fdr_sig/total_tested*100:.1f}%)")

# 3. 查看你文章里讨论的核心互作（例如 TGFB1, ICAM1, VEGFA, VCAM1 等）的 padj 是否显著
key_molecules = ['TGFB', 'ICAM1', 'VCAM1', 'VEGF', 'CCL2', 'CXCL']
check_sub = ec_mp_cross_res[
    ec_mp_cross_res['ligand_complex'].str.contains('|'.join(key_molecules), na=False) |
    ec_mp_cross_res['receptor_complex'].str.contains('|'.join(key_molecules), na=False)
]
print("\n部分核心配受体校正后的 padj 样例：")
print(check_sub[['time_point', 'direction', 'ligand_complex', 'receptor_complex', 'cellphone_pvals', 'cellphone_padj', 'magnitude_rank']].head(10))

# 4. 重新导出更新后的结果文件作为 Supplementary Table
ec_mp_cross_res.to_csv("PigEC_HumanMP_ortholog_liana_results_with_FDR.csv", index=False)
print("\n已保存包含 FDR 校正结果的文件: PigEC_HumanMP_ortholog_liana_results_with_FDR.csv")

In [ ]:
# 筛选出 FDR < 0.05 的 50 个互作看看都有谁
sig_50 = ec_mp_cross_res[ec_mp_cross_res['cellphone_padj'] < 0.05]
print("=== FDR < 0.05 的核心互作列表（前 20 个）===")
print(sig_50[['time_point', 'direction', 'ligand_complex', 'receptor_complex', 'cellphone_pvals', 'cellphone_padj']].head(20))

# 看看 pvals < 0.05 但 padj >= 0.05 的互作（46 个）
borderline = ec_mp_cross_res[(ec_mp_cross_res['cellphone_pvals'] < 0.05) & (ec_mp_cross_res['cellphone_padj'] >= 0.05)]
print("\n=== 名义显著但被 FDR 过滤的互作（前 10 个）===")
print(borderline[['time_point', 'direction', 'ligand_complex', 'receptor_complex', 'cellphone_pvals', 'cellphone_padj']].head(10))

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from matplotlib.ticker import MaxNLocator

# =========================================================
# 1. 严格基于 FDR < 0.05 筛选显著相互作用
# =========================================================
print("1. 正在基于 FDR < 0.05 统计各时间点跨物种通讯数量...")

# 使用我们刚才添加了 cellphone_padj 的 ec_mp_cross_res 表
ec_mp_fdr_sig = ec_mp_cross_res[ec_mp_cross_res['cellphone_padj'] < 0.05].copy()

mp_cell_name = "MP_total"
ec_cell_name = "EC_total"

# 两个方向的掩码
mp_to_ec_mask = (ec_mp_fdr_sig['source'] == mp_cell_name) & (ec_mp_fdr_sig['target'] == ec_cell_name)
ec_to_mp_mask = (ec_mp_fdr_sig['source'] == ec_cell_name) & (ec_mp_fdr_sig['target'] == mp_cell_name)

ec_mp_fdr_sig_cross = ec_mp_fdr_sig[mp_to_ec_mask | ec_to_mp_mask].copy()

ec_mp_fdr_sig_cross['Direction'] = np.where(
    ec_mp_fdr_sig_cross['source'] == mp_cell_name,
    "Human MP $\\rightarrow$ Pig EC",
    "Pig EC $\\rightarrow$ Human MP"
)

# 分组计数
df_counts_raw = ec_mp_fdr_sig_cross.groupby(['time_point', 'Direction']).size().reset_index(name='Significant_Interactions')

# 补齐所有组合
tps = ["D14", "D33", "D61"]
dirs = ["Human MP $\\rightarrow$ Pig EC", "Pig EC $\\rightarrow$ Human MP"]
full_idx = pd.MultiIndex.from_product([tps, dirs], names=['time_point', 'Direction'])
df_counts = df_counts_raw.set_index(['time_point', 'Direction']).reindex(full_idx, fill_value=0).reset_index()

print("FDR 校正后的柱状图统计数据：")
print(df_counts)

# =========================================================
# 2. 绘制图A (Barplot)
# =========================================================
plt.figure(figsize=(4.8, 4.5))
sns.set_theme(style="ticks")

palette_colors = {
    "Human MP $\\rightarrow$ Pig EC": "#08457E",  # 深蓝
    "Pig EC $\\rightarrow$ Human MP": "#4B9CD3"   # 中蓝
}

df_counts["time_point"] = pd.Categorical(df_counts["time_point"], categories=["D14", "D33", "D61"], ordered=True)

ax = sns.barplot(
    data=df_counts,
    x="time_point",
    y="Significant_Interactions",
    hue="Direction",
    palette=palette_colors,
    edgecolor="#333333",
    linewidth=1.0
)

# 关键修改：明确标明是 Adjusted P < 0.05 / FDR < 0.05
plt.title("Significant Cross-species Interactions\n(FDR < 0.05)", fontsize=11, fontweight='bold', pad=12)
plt.ylabel("Significant LR Pairs (FDR < 0.05)", fontsize=10.5, fontweight='bold')
plt.xlabel("Time Point (Days Post-transplantation)", fontsize=10.5, fontweight='bold')

plt.legend(
    title="Direction",
    bbox_to_anchor=(1.02, 1),
    loc='upper left',
    frameon=False,
    fontsize=9.5,
    title_fontsize=10.5
)

sns.despine()
ax.yaxis.set_major_locator(MaxNLocator(integer=True))

plt.tight_layout()
plt.savefig("FigA_Interaction_Counts_FDR_corrected.pdf", dpi=300, bbox_inches="tight")
plt.savefig("FigA_Interaction_Counts_FDR_corrected.png", dpi=300, bbox_inches="tight")
plt.show()
print("-> 图A已保存: FigA_Interaction_Counts_FDR_corrected.pdf")

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# =========================================================
# 3. 总体通讯：基于 FDR < 0.05 的信号通路归类与时序准备
# =========================================================
print("3. 正在对 FDR < 0.05 的显著通讯进行高置信信号通路分类...")

# 1. 严格筛选 FDR 校正后显著的数据 (cellphone_padj < 0.05)
ec_mp_sig_fdr = ec_mp_cross_res[ec_mp_cross_res['cellphone_padj'] < 0.05].copy()

# 确保时间点时序正确
ec_mp_sig_fdr["time_point"] = pd.Categorical(ec_mp_sig_fdr["time_point"], categories=["D14", "D33", "D61"], ordered=True)
ec_mp_sig_fdr["LR_pair"] = ec_mp_sig_fdr["ligand_complex"] + " $\\rightarrow$ " + ec_mp_sig_fdr["receptor_complex"]

# 2. 精细化定制通路词典（彻底移除争议的 Metabolic / Calcium 假阳性组）
# 针对校正后真实显著的分子：TGFB1, SEMA3B, SEMA4A, PDGFC, COL4A1/2, COL5A2, EFNB2, CDH1 等
def get_pathway_group(lr_pair):
    pair_upper = lr_pair.upper()
    pathway_rules = {
        "TGF-b / BMP Signaling": ["TGFB", "ENG", "INHBC", "ACVR", "BMP", "CAV1"],
        "Semaphorin Guidance": ["SEMA", "NRP1", "NRP2", "PLXN", "PLXNA"],
        "Growth Factor / Angiogenesis": ["PDGF", "FLT", "KDR", "VEGF", "EGF", "FGF", "NAMPT", "INSR"],
        "Adhesion / ECM / Matrix": ["COL4", "COL5", "CD44", "DDR", "CDH", "PTPRM", "EFNB", "EPHB", "VCAN", "TLR", "ITG", "ICAM", "VCAM", "LRP1B", "PLAUR", "NCAM", "ROBO", "CNTN"],
        "Chemokine / Cytokine": ["CXCL", "CCL", "CXCR", "CCR", "ACKR"],
        "Complement System": ["C3", "C5", "C3AR", "C5AR", "CFB", "CD46"]
    }
    for pathway, keywords in pathway_rules.items():
        if any(kw in pair_upper for kw in keywords):
            return pathway
    return "Other Significant Pathways"

ec_mp_sig_fdr["Pathway"] = ec_mp_sig_fdr["LR_pair"].apply(get_pathway_group)

# 3. 区分通信方向（直接利用数据里现成标准的 direction 列）
# 方向1：MP发送给EC (MP -> EC)
mp_to_ec_total = ec_mp_sig_fdr[ec_mp_sig_fdr["direction"] == "MP_total_to_EC_total"].copy()

# 方向2：EC发送给MP (EC -> MP)
ec_to_mp_total = ec_mp_sig_fdr[ec_mp_sig_fdr["direction"] == "EC_total_to_MP_total"].copy()

print(f"-> 筛选与分类完成！(FDR < 0.05 严格标准)")
print(f"   Human MP -> Pig EC 共有 {len(mp_to_ec_total)} 条显著记录；")
print(f"   Pig EC -> Human MP 共有 {len(ec_to_mp_total)} 条显著记录。")

# 4. 打印查看两个方向在各个时间点和通路的分布概况
print("\n[Human MP -> Pig EC] 通路与时间点分布：")
print(pd.crosstab(mp_to_ec_total['Pathway'], mp_to_ec_total['time_point']))

print("\n[Pig EC -> Human MP] 通路与时间点分布：")
print(pd.crosstab(ec_to_mp_total['Pathway'], ec_to_mp_total['time_point']))

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# =========================================================
# 方案一：总体通路时序热图 (基于 FDR < 0.05，缺失值优雅留灰)
# =========================================================
print("正在绘制基于 FDR < 0.05 的总体通路时序热图...")

# 1. 自动适配分数指标列 (兼容 lr_means / lrscore / expr_prod)
candidate_score_cols = ["lrscore", "lr_means", "mean", "expr_prod", "magnitude_rank"]
score_col = None
for col in candidate_score_cols:
    if col in mp_to_ec_total.columns:
        score_col = col
        break

if score_col is None:
    raise ValueError(f"未找到可用的通讯强度列，当前列名包括: {list(mp_to_ec_total.columns)}")

print(f"-> 正在使用指标列: '{score_col}' 计算通路平均活性")

# 2. 计算各时间点通路的平均分 (仅针对 FDR < 0.05 的高置信互作)
trend_df = (
    mp_to_ec_total.groupby(["time_point", "Pathway"], observed=True)[score_col]
    .mean()
    .reset_index()
)

# 过滤掉杂项通路
trend_df = trend_df[~trend_df["Pathway"].isin(["Other Pathways", "Other Significant Pathways"])].copy()

# 3. 数据透视，并规范化时序轴与通路顺序
heatmap_data = trend_df.pivot(index="Pathway", columns="time_point", values=score_col)

# 确保三大核心时间点全部存在
for tp in ["D14", "D33", "D61"]:
    if tp not in heatmap_data.columns:
        heatmap_data[tp] = np.nan
heatmap_data = heatmap_data[["D14", "D33", "D61"]]

# 推荐的学术生物学展示顺序（从生长因子/TGF-b 到轴突导向、细胞外基质、趋化因子）
preferred_order = [
    "TGF-b / BMP Signaling",
    "Semaphorin Guidance",
    "Growth Factor / Angiogenesis",
    "Adhesion / ECM / Matrix",
    "Chemokine / Cytokine",
    "Complement System"
]
existing_order = [p for p in preferred_order if p in heatmap_data.index]
# 若有其他未列入 preferred_order 的分类则追加在末尾
remaining_order = [p for p in heatmap_data.index if p not in existing_order]
final_order = existing_order + remaining_order
heatmap_data = heatmap_data.reindex(final_order)

# 4. 开始绘制热图
plt.figure(figsize=(5.5, 4.6))
sns.set_theme(style="white")

# 设定热图颜色上下限
valid_values = heatmap_data.values[~np.isnan(heatmap_data.values)]
vmin = np.nanmin(valid_values) * 0.9 if len(valid_values) > 0 else 0
vmax = np.nanmax(valid_values) * 1.05 if len(valid_values) > 0 else 1

ax = sns.heatmap(
    heatmap_data,
    cmap="Blues",
    annot=True,                 # 显示具体数值
    fmt=".2f",                  # 保留两位小数
    linewidths=1.5,
    linecolor='white',
    cbar_kws={
        "shrink": 0.75,
        "label": f"Mean Strength ({score_col})"
    },
    mask=heatmap_data.isnull(), # 自动隐藏无显著数据的格子
    vmin=vmin,
    vmax=vmax
)

# 将无数据的格子背景设为低饱和度浅灰，代表未达到 FDR < 0.05 阈值
ax.set_facecolor('#EEEEEE')

# 加上清晰的四周边框线
for _, spine in ax.spines.items():
    spine.set_visible(True)
    spine.set_color('#333333')
    spine.set_linewidth(1.2)

# 设置轴标签与标题
plt.title(
    "Macro-Pathway Activity Over Time\n(FDR < 0.05 Core Set: Total MP $\\rightarrow$ Total EC)",
    fontsize=11.5,
    fontweight='bold',
    pad=15
)
plt.xlabel("Time Point (Days Post-transplantation)", fontsize=10.5, fontweight='bold', labelpad=8)
plt.ylabel("Signaling Family", fontsize=10.5, fontweight='bold', labelpad=8)
plt.xticks(fontsize=11, fontweight='bold')
plt.yticks(fontsize=10)

plt.tight_layout()
plt.savefig("FigC_Total_MP_to_EC_Pathway_Heatmap_FDR.pdf", dpi=300, bbox_inches="tight")
plt.savefig("FigC_Total_MP_to_EC_Pathway_Heatmap_FDR.png", dpi=300, bbox_inches="tight")
plt.show()

print("-> 已保存 FDR 矫正后的总体通路热图：FigC_Total_MP_to_EC_Pathway_Heatmap_FDR.pdf")

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# =========================================================
# 1. 自动适配打分列名 (兼容 lrscore / lr_means / expr_prod)
# =========================================================
candidate_score_cols = ["lrscore", "lr_means", "mean", "expr_prod"]
score_col = None
for col in candidate_score_cols:
    if col in mp_to_ec_total.columns:
        score_col = col
        break

if score_col is None:
    raise ValueError(f"未找到可用的通讯强度列，请检查 mp_to_ec_total 的列名: {list(mp_to_ec_total.columns)}")

print(f"-> 气泡图正在使用通讯强度列: '{score_col}'")

# =========================================================
# 2. 数据准备与排序（移除假阳性代谢组，保留核心生物学通路）
# =========================================================
pathway_order = [
    "TGF-b / BMP Signaling",
    "Chemokine / Cytokine",
    "Semaphorin Guidance",
    "Growth Factor / Angiogenesis",
    "Adhesion / ECM / Matrix",
    "Complement System"
]

# 筛选属于上述核心通路且通过 FDR < 0.05 的数据
sub_df = mp_to_ec_total[mp_to_ec_total["Pathway"].isin(pathway_order)].copy()

# 筛选每个通路最具代表性的前 4 个配受体对（按平均强度降序）
top_pairs = (
    sub_df.groupby(["Pathway", "LR_pair"], observed=True)[score_col]
    .mean()
    .reset_index()
    .sort_values(["Pathway", score_col], ascending=[True, False])
    .groupby("Pathway", observed=True)
    .head(4)
)

plot_data = sub_df[sub_df["LR_pair"].isin(top_pairs["LR_pair"])].copy()

# 计算唯一的 (Pathway, LR_pair) 组合并映射到固定的 Y 轴坐标
unique_rows = plot_data[["Pathway", "LR_pair"]].drop_duplicates().copy()
unique_rows["Pathway"] = pd.Categorical(unique_rows["Pathway"], categories=pathway_order, ordered=True)

# 排序：按 pathway_order 顺序排列，再倒序，使第一个通路画在最顶端
unique_rows = unique_rows.sort_values(["Pathway", "LR_pair"], ascending=[True, True])
unique_rows = unique_rows.iloc[::-1].reset_index(drop=True)
unique_rows["y_idx"] = unique_rows.index

# 合并 Y 坐标回主绘图数据
plot_data = plot_data.merge(unique_rows[["Pathway", "LR_pair", "y_idx"]], on=["Pathway", "LR_pair"])

# 将时间点稳定映射为数值 X 坐标
tp_map = {"D14": 0, "D33": 1, "D61": 2}
plot_data["x_idx"] = plot_data["time_point"].astype(str).map(tp_map)

# =========================================================
# 3. 开始绘图 (画布高度根据保留的分子数自适应)
# =========================================================
n_pairs = len(unique_rows)
fig_height = max(6.5, n_pairs * 0.42 + 2.0)
fig, ax = plt.subplots(figsize=(6.5, fig_height))
sns.set_theme(style="ticks")

# 设定 X/Y 轴范围
ax.set_xlim(-0.4, 2.4)
ax.set_ylim(-0.6, n_pairs - 0.4)

# 绘制交替背景色带 (Shading)
pathways_in_data = [p for p in pathway_order if p in unique_rows["Pathway"].values]
for idx, pw in enumerate(pathways_in_data):
    pw_rows = unique_rows[unique_rows["Pathway"] == pw]
    min_y = pw_rows["y_idx"].min()
    max_y = pw_rows["y_idx"].max()
    
    # 奇数通路行填充极浅灰色，偶数行保留白色
    if idx % 2 == 1:
        ax.axhspan(min_y - 0.5, max_y + 0.5, facecolor='#F4F6F7', alpha=0.7, zorder=0)

# 绘制通路之间的水平分割线
for i in range(len(unique_rows) - 1):
    if unique_rows.loc[i, "Pathway"] != unique_rows.loc[i+1, "Pathway"]:
        ax.axhline(y=i+0.5, color="#BDC3C7", linestyle="-", linewidth=0.9, zorder=1)

# 气泡大小自适应缩放因子
max_val = plot_data[score_col].max()
scale_factor = 450.0 / max_val if max_val > 0 else 300.0

# 绘制气泡
scatter = ax.scatter(
    x=plot_data["x_idx"],
    y=plot_data["y_idx"],
    s=plot_data[score_col] * scale_factor,
    c=plot_data[score_col],
    cmap="Blues",
    edgecolors="#1A252C",
    linewidths=0.9,
    zorder=3
)

# =========================================================
# 4. 绘制右侧学术大括号与通路名称
# =========================================================
for pw in pathways_in_data:
    pw_rows = unique_rows[unique_rows["Pathway"] == pw]
    min_y = pw_rows["y_idx"].min()
    max_y = pw_rows["y_idx"].max()
    mid_y = (min_y + max_y) / 2.0
    
    # 括号几何坐标
    x_bracket = 2.48
    x_tick = 2.43
    y_start = min_y - 0.35
    y_end = max_y + 0.35
    
    # 画括号骨干与上下横突
    ax.plot([x_bracket, x_bracket], [y_start, y_end], color="#5D6D7E", linewidth=1.5, clip_on=False)
    ax.plot([x_tick, x_bracket], [y_start, y_start], color="#5D6D7E", linewidth=1.5, clip_on=False)
    ax.plot([x_tick, x_bracket], [y_end, y_end], color="#5D6D7E", linewidth=1.5, clip_on=False)
    
    # 通路文本标注
    ax.text(
        x_bracket + 0.08,
        mid_y,
        pw,
        va='center',
        ha='left',
        fontsize=10.5,
        fontweight='bold',
        color='#1A365D',
        clip_on=False
    )

# =========================================================
# 5. 轴标签与细节美化
# =========================================================
# Y 轴：配受体名称
ax.set_yticks(unique_rows["y_idx"])
ax.set_yticklabels(unique_rows["LR_pair"], fontsize=10.5)

# X 轴：时间点
ax.set_xticks([0, 1, 2])
ax.set_xticklabels(["D14", "D33", "D61"], fontsize=11, fontweight='bold')
ax.set_xlabel("Time Point (Days Post-transplantation)", fontsize=11, fontweight='bold', labelpad=10)

# 网格与边框
ax.grid(axis='x', linestyle=':', alpha=0.6, color='#BDC3C7')
sns.despine(top=True, right=True, left=False, bottom=False)

# 标题
plt.title(
    "Key Ligand–Receptor Pairs Driving Pathway Dynamics\n(FDR < 0.05: Total MP $\\rightarrow$ Total EC)",
    fontsize=12,
    fontweight='bold',
    pad=18
)

# =========================================================
# 6. 图例设置（下方横排）
# =========================================================
handles, labels = scatter.legend_elements(
    prop="sizes", 
    alpha=0.6, 
    num=4, 
    func=lambda x: x / scale_factor
)
legend = ax.legend(
    handles,
    labels,
    loc="upper center",
    bbox_to_anchor=(0.5, -0.08),
    ncol=4,
    frameon=False,
    title=f"Mean Strength ({score_col})",
    title_fontsize=10.5
)
ax.add_artist(legend)

# 保存高清 PDF 与 PNG
plt.tight_layout()
plt.savefig("FigD_Total_MP_to_EC_FDR_Key_Pairs.pdf", dpi=300, bbox_inches="tight")
plt.savefig("FigD_Total_MP_to_EC_FDR_Key_Pairs.png", dpi=300, bbox_inches="tight")
plt.show()

print("-> 已保存 FDR 矫正后的时序关键配受体气泡图：FigD_Total_MP_to_EC_FDR_Key_Pairs.pdf")

In [ ]:
import liana as li
import pandas as pd
import numpy as np
from statsmodels.stats.multitest import multipletests

# =======================================================
# 1. 亚群 LIANA 计算 (D14, D33, D61)
# =======================================================
subcluster_col = "celltype_for_liana" 

main_time_points = ["D14", "D33", "D61"] 
adata_main = adata_cross[adata_cross.obs["time_point"].isin(main_time_points)].copy() 
results_by_time_sub = {} 

for tp in main_time_points: 
    print(f"\n==========================================")
    print(f"Running Sub-cluster LIANA for {tp}...") 
    print(f"==========================================")
    
    adata_tp = adata_main[adata_main.obs["time_point"] == tp].copy() 
    
    # 打印当前的亚群分布
    print(pd.crosstab(adata_tp.obs["time_point"], adata_tp.obs[subcluster_col])) 

    # 按亚群列运算
    li.mt.rank_aggregate( 
        adata_tp, 
        groupby=subcluster_col, 
        resource_name="consensus", 
        expr_prop=0.1, 
        min_cells=10, 
        use_raw=False, 
        verbose=True
    ) 

    res = adata_tp.uns["liana_res"].copy() 
    res["time_point"] = tp
    results_by_time_sub[tp] = res

# 合并三个时间点的全部亚群结果
df_all_sub = pd.concat(results_by_time_sub.values(), ignore_index=True)

# =======================================================
# 2. 全局多重检验校正 (FDR / Benjamini-Hochberg)
# =======================================================
print("\n正在对亚群通讯结果进行全局 Benjamini-Hochberg FDR 校正...")

# 找到 CellPhoneDB 原始置换检验 p-value 列
pval_col = 'cellphone_pvals' if 'cellphone_pvals' in df_all_sub.columns else 'pvals'

# 进行 BH 校正
reject, padj, _, _ = multipletests(df_all_sub[pval_col], alpha=0.05, method='fdr_bh')
df_all_sub['cellphone_padj'] = padj

# =======================================================
# 3. 跨物种亚群通讯精准识别 (Human MP 亚群 <-> Pig EC 亚群)
# =======================================================
# 自动/手动识别哪些亚群属于 MP，哪些属于 EC
# 请确认你的亚群命名规则，这里支持包含关键字的自动归类：
all_clusters = df_all_sub['source'].unique().tolist()
mp_subs = [c for c in all_clusters if any(k in c.upper() for k in ["MP", "MAC", "MONO", "MDM"])]
ec_subs = [c for c in all_clusters if any(k in c.upper() for k in ["EC", "ENDO", "SEC", "AEC", "VEC", "CAP"])]

print(f"\n自动识别的 MP 亚群: {mp_subs}")
print(f"自动识别的 EC 亚群: {ec_subs}")

# 筛选跨物种双向通讯
mp_to_ec_mask = (df_all_sub['source'].isin(mp_subs)) & (df_all_sub['target'].isin(ec_subs))
ec_to_mp_mask = (df_all_sub['source'].isin(ec_subs)) & (df_all_sub['target'].isin(mp_subs))

sub_cross_res = df_all_sub[mp_to_ec_mask | ec_to_mp_mask].copy()

# 标记方向
sub_cross_res['direction'] = np.where(
    sub_cross_res['source'].isin(mp_subs),
    "MP_sub_to_EC_sub",
    "EC_sub_to_MP_sub"
)

# 筛选通过 FDR < 0.05 的真显著条目
sub_cross_sig = sub_cross_res[sub_cross_res['cellphone_padj'] < 0.05].copy()

print(f"\n-> 亚群级别统计汇总:")
print(f"   总跨物种亚群测试数: {len(sub_cross_res)}")
print(f"   名义显著 (p < 0.05): {(sub_cross_res[pval_col] < 0.05).sum()}")
print(f"   FDR 严格显著 (padj < 0.05): {len(sub_cross_sig)}")

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# =========================================================
# 1. 确保细胞亚群定义并基于 FDR < 0.05 筛选
# =========================================================
print("正在绘制基于 FDR < 0.05 的所有 EC 亚群时序通讯折线图...")

mp_subclusters = ["FCN1+MP"] 
ec_subclusters = [ 
    "Arterial EC", 
    "ECM-remodeling Arterial EC", 
    "Venous EC", 
    "Capillary EC_1", 
    "Capillary EC_2", 
    "Glomerular EC" 
] 
time_points = ["D14", "D33", "D61"]

# 使用上一步 FDR 校正后的亚群结果表 df_all_sub
# 严格过滤: cellphone_padj < 0.05
sub_sig_fdr = df_all_sub[df_all_sub["cellphone_padj"] < 0.05].copy()

# =========================================================
# 2. 提取并计算两个方向的时序通讯总数 (严谨补 0)
# =========================================================
# 创建完整索引，防止某亚群在某时间点没有显著互作导致断线
full_index = pd.MultiIndex.from_product(
    [time_points, ec_subclusters], 
    names=["time_point", "EC_Subcluster"]
)

# --- 方向一：MP -> EC (EC 作为接收端 Target) --- 
mp_to_ec_raw = sub_sig_fdr[ 
    sub_sig_fdr["source"].isin(mp_subclusters) & sub_sig_fdr["target"].isin(ec_subclusters) 
]
mp_to_ec_counts = (
    mp_to_ec_raw.groupby(["time_point", "target"], observed=False)
    .size()
    .rename_axis(["time_point", "EC_Subcluster"])
    .reindex(full_index, fill_value=0)
    .reset_index(name="count")
)

# --- 方向二：EC -> MP (EC 作为发送端 Source) --- 
ec_to_mp_raw = sub_sig_fdr[ 
    sub_sig_fdr["source"].isin(ec_subclusters) & sub_sig_fdr["target"].isin(mp_subclusters) 
]
ec_to_mp_counts = (
    ec_to_mp_raw.groupby(["time_point", "source"], observed=False)
    .size()
    .rename_axis(["time_point", "EC_Subcluster"])
    .reindex(full_index, fill_value=0)
    .reset_index(name="count")
)

# =========================================================
# 3. 开始绘制折线图 (1行2列并排展示) 
# =========================================================
fig, axes = plt.subplots(1, 2, figsize=(13, 5.5), sharey=True) 
sns.set_theme(style="ticks") 

# 配色与标记样式
ec_colors = { 
    "Arterial EC": "#E74C3C",                # 红色
    "ECM-remodeling Arterial EC": "#F39C12",  # 橙色
    "Venous EC": "#3498DB",                  # 蓝色
    "Capillary EC_1": "#2ECC71",             # 绿色
    "Capillary EC_2": "#27AE60",             # 深绿
    "Glomerular EC": "#9B59B6"               # 紫色
} 

markers = { 
    "Arterial EC": "o", 
    "ECM-remodeling Arterial EC": "s", 
    "Venous EC": "^", 
    "Capillary EC_1": "v", 
    "Capillary EC_2": "D", 
    "Glomerular EC": "p" 
} 

# --------------------------------------------------------- 
# 子图 1：MP -> EC (EC 接收的通讯数) 
# --------------------------------------------------------- 
ax1 = axes[0] 
for ec in ec_subclusters: 
    sub_data = mp_to_ec_counts[mp_to_ec_counts["EC_Subcluster"] == ec] 
    ax1.plot( 
        sub_data["time_point"], 
        sub_data["count"], 
        marker=markers[ec], 
        color=ec_colors[ec], 
        linewidth=2.5, 
        markersize=8, 
        label=ec
    ) 

ax1.set_title("Incoming Signals\n(FCN1+MP $\\rightarrow$ EC Subclusters)", fontsize=12.5, fontweight='bold', pad=12) 
ax1.set_ylabel("Significant Interactions (FDR < 0.05)", fontsize=11, fontweight='bold') 
ax1.set_xlabel("Time Point (Days Post-transplantation)", fontsize=11, fontweight='bold') 
ax1.grid(axis='both', linestyle=':', alpha=0.5) 
ax1.set_xticks([0, 1, 2])
ax1.set_xticklabels(time_points, fontsize=11, fontweight='bold')

# --------------------------------------------------------- 
# 子图 2：EC -> MP (EC 发送的通讯数) 
# --------------------------------------------------------- 
ax2 = axes[1] 
for ec in ec_subclusters: 
    sub_data = ec_to_mp_counts[ec_to_mp_counts["EC_Subcluster"] == ec] 
    ax2.plot( 
        sub_data["time_point"], 
        sub_data["count"], 
        marker=markers[ec], 
        color=ec_colors[ec], 
        linewidth=2.5, 
        markersize=8, 
        label=ec
    ) 

ax2.set_title("Outgoing Signals\n(EC Subclusters $\\rightarrow$ FCN1+MP)", fontsize=12.5, fontweight='bold', pad=12) 
ax2.set_xlabel("Time Point (Days Post-transplantation)", fontsize=11, fontweight='bold') 
ax2.grid(axis='both', linestyle=':', alpha=0.5) 
ax2.set_xticks([0, 1, 2])
ax2.set_xticklabels(time_points, fontsize=11, fontweight='bold')

# =========================================================
# 4. 细节调整与图例美化
# =========================================================
sns.despine(fig, top=True, right=True) 

# 将图例统一放置在右侧
handles, labels = ax1.get_legend_handles_labels() 
fig.legend( 
    handles, 
    labels, 
    loc="center right", 
    bbox_to_anchor=(1.16, 0.5), 
    frameon=False, 
    title="EC Subclusters", 
    title_fontsize=11.5, 
    fontsize=10.5
) 

plt.suptitle("Temporal Dynamics of Cross-species Interactions Across EC Subclusters\n(Benjamini–Hochberg FDR < 0.05)", 
             fontsize=13.5, fontweight='bold', y=1.04) 

plt.tight_layout() 
plt.subplots_adjust(right=0.86) 

# 保存 PDF 与 PNG
plt.savefig("All_EC_Temporal_Linecharts_FDR.pdf", dpi=300, bbox_inches="tight") 
plt.savefig("All_EC_Temporal_Linecharts_FDR.png", dpi=300, bbox_inches="tight") 
plt.show() 

print("-> 已保存 FDR 严格校正后的 EC 亚群时序折线图：All_EC_Temporal_Linecharts_FDR.pdf")

In [ ]:
import pandas as pd
import numpy as np

# =========================================================
# 1. 定义发送端与接收端，对接 FDR < 0.05 的亚群数据
# =========================================================
fcn1_mp = ["FCN1+MP"] 
ec_subclusters = [ 
    "Arterial EC", "ECM-remodeling Arterial EC", "Venous EC", 
    "Capillary EC_1", "Capillary EC_2", "Glomerular EC" 
] 

# 严格提取：FCN1+MP -> EC 亚群 且 cellphone_padj < 0.05
df_check = df_all_sub[ 
    (df_all_sub["source"].isin(fcn1_mp)) & 
    (df_all_sub["target"].isin(ec_subclusters)) &
    (df_all_sub["cellphone_padj"] < 0.05)
].copy() 

# 确保时间点时序
df_check["time_point"] = pd.Categorical(df_check["time_point"], categories=["D14", "D33", "D61"], ordered=True)

# =========================================================
# 2. 拼接配体和受体，生成唯一的交互名称
# =========================================================
df_check["interacting_pair"] = df_check["ligand_complex"].astype(str) + " → " + df_check["receptor_complex"].astype(str) 

# 提取并打印唯一交互对
unique_pairs = sorted(df_check["interacting_pair"].dropna().unique())

print(f"=== [FDR < 0.05 严格筛选] FCN1+MP -> ECs 方向共确认 {len(unique_pairs)} 种高置信受体-配体对 ===") 
print(f"前 30 个高置信交互对预览:\n{unique_pairs[:30]}") 
print("===================================================================\n") 

# =========================================================
# 3. 统计汇总表 A：每个受体-配体对在各时间点命中的 EC 亚群数量 (Breadth)
# =========================================================
summary_counts = pd.crosstab(df_check["interacting_pair"], df_check["time_point"]) 
summary_counts.columns.name = "Time Point" 
summary_counts.index.name = "Interacting Pair (FDR < 0.05)" 

print("--- 每个受体-配体对在各时间点覆盖的 EC 亚群频次预览 ---") 
print(summary_counts.head(10)) 

# 保存频次统计表
summary_counts.to_csv("Supplementary_Table_FCN1MP_to_EC_Pair_Counts_FDR.csv") 
print("\n-> 亚群频次表已保存为：Supplementary_Table_FCN1MP_to_EC_Pair_Counts_FDR.csv") 

# =========================================================
# 4. 统计汇总表 B：每个受体-配体对在各时间点的【平均通讯强度】(Intensity)
# =========================================================
score_col = 'lr_means' if 'lr_means' in df_check.columns else ('expr_prod' if 'expr_prod' in df_check.columns else 'lrscore')

summary_strength = (
    df_check.groupby(["interacting_pair", "time_point"], observed=False)[score_col]
    .mean()
    .unstack(level="time_point")
)
summary_strength.to_csv("Supplementary_Table_FCN1MP_to_EC_Mean_Strength_FDR.csv")
print(f"-> 平均通讯强度透视表已保存为：Supplementary_Table_FCN1MP_to_EC_Mean_Strength_FDR.csv")

# =========================================================
# 5. 导出完整明细表 (供审稿人查阅的完整 Supporting Data)
# =========================================================
detail_cols = [ 
    "time_point", "source", "target", 
    "ligand_complex", "receptor_complex", "interacting_pair", 
    score_col, "cellphone_pvals", "cellphone_padj"
] 
# 动态保留存在的列，防止 KeyError
existing_cols = [col for col in detail_cols if col in df_check.columns] 

# 按照时间点、配对名、校正后 padj 升序排列
df_detail = df_check[existing_cols].sort_values(
    by=["time_point", "interacting_pair", "cellphone_padj"], 
    ascending=[True, True, True]
) 

df_detail.to_csv("Supplementary_Table_FCN1MP_to_EC_Full_Detail_FDR.csv", index=False) 
print("-> 完整交互明细表（含 FDR padj）已保存为：Supplementary_Table_FCN1MP_to_EC_Full_Detail_FDR.csv")

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# =========================================================
# 1. 自动适配打分列名与数据准备 (严格基于 FDR < 0.05)
# =========================================================
print("正在绘制基于 FDR < 0.05 的 FCN1+MP -> 各 EC 亚群通路热图网格...")

candidate_score_cols = ["lrscore", "lr_means", "mean", "expr_prod"]
score_col = None
for col in candidate_score_cols:
    if col in df_all_sub.columns:
        score_col = col
        break

if score_col is None:
    raise ValueError(f"未找到可用的通讯强度列，请检查 df_all_sub 的列名: {list(df_all_sub.columns)}")

print(f"-> 正在使用指标列: '{score_col}'")

# 严格提取：FCN1+MP 作为发送端，靶细胞为 EC，且通过 Benjamini-Hochberg 校正
fcn1_mp = ["FCN1+MP"] 
df_fcn1_to_ec = df_all_sub[ 
    (df_all_sub["source"].isin(fcn1_mp)) & 
    (df_all_sub["target"].str.contains("EC", na=False)) &
    (df_all_sub["cellphone_padj"] < 0.05)
].copy() 

# 宏观家族生物学规范分类函数
def categorize_pathway(ligand): 
    p = str(ligand).upper() 
    if any(x in p for x in ["CXCL", "CCL", "IL", "TNF", "IFNG", "LTA", "CNTF", "CSF", "OSM"]): 
        return "Chemokine / Cytokine" 
    elif any(x in p for x in ["COMPLEMENT", "C3", "C5", "CFD", "CFB", "C1Q"]): 
        return "Complement System" 
    elif any(x in p for x in ["VEGF", "FGF", "PDGF", "ANGPT", "EGF", "IGF", "HGF", "JAG", "EDN", "EPGN"]): 
        return "Growth Factor / Angiogenesis" 
    elif any(x in p for x in ["SEMA", "EPH", "EFN", "SLIT", "ROBO", "PLXN"]): 
        return "Semaphorin & Guidance" 
    elif any(x in p for x in ["TGF", "BMP", "GDF", "INHBA", "GREM", "ENG", "ACV"]): 
        return "TGF-b / BMP Signaling" 
    elif any(x in p for x in ["COL", "LAM", "FN1", "CD99", "PECAM", "ICAM", "VCAM", "MMP", "CDH", "DCN", "THBS"]): 
        return "Adhesion / ECM / Matrix" 
    else: 
        return "Other Significant Pathways" 

df_fcn1_to_ec['Signaling Family'] = df_fcn1_to_ec['ligand_complex'].apply(categorize_pathway) 

# =========================================================
# 2. 准备统一的坐标轴和色带标准
# =========================================================
ordered_times = ["D14", "D33", "D61"] 
ordered_families = [ 
    "TGF-b / BMP Signaling", 
    "Semaphorin & Guidance",
    "Growth Factor / Angiogenesis", 
    "Adhesion / ECM / Matrix", 
    "Chemokine / Cytokine", 
    "Complement System"
] 

# 过滤掉杂项通路（使结论集中在经典生物学通路）
df_plot = df_fcn1_to_ec[df_fcn1_to_ec['Signaling Family'].isin(ordered_families)].copy()

# 找到实际存在的 EC 亚群并按逻辑排序
targets = sorted(df_plot['target'].unique()) 
n_targets = len(targets) 

# 计算全局的均值上下限，用于统一所有子图的颜色深浅对比
global_mean = df_plot.groupby(['target', 'time_point', 'Signaling Family'], observed=False)[score_col].mean() 
valid_means = global_mean.dropna().values
vmax_global = np.percentile(valid_means, 98) if len(valid_means) > 0 else 1.0
vmin_global = np.min(valid_means) * 0.9 if len(valid_means) > 0 else 0.0

# =========================================================
# 3. 动态计算画板大小并循环画图
# =========================================================
cols = 3  # 每行放3个图
rows = (n_targets + cols - 1) // cols
fig, axes = plt.subplots(rows, cols, figsize=(4.8 * cols, 4.2 * rows), squeeze=False) 
axes = axes.flatten()  # 展平以便于循环

for i, target_ec in enumerate(targets): 
    ax = axes[i] 
    
    # 提取当前 EC 的数据并计算通路平均强度
    df_sub = df_plot[df_plot['target'] == target_ec] 
    heat_data = (
        df_sub.groupby(['time_point', 'Signaling Family'], observed=False)[score_col]
        .mean()
        .reset_index()
    ) 
    pivot_df = heat_data.pivot(index='Signaling Family', columns='time_point', values=score_col) 
    
    # 补全缺失的时间点和通路，确保所有子图结构完全一致
    for t in ordered_times: 
        if t not in pivot_df.columns: 
            pivot_df[t] = np.nan
    pivot_df = pivot_df[ordered_times].reindex(ordered_families) 
    
    # 判断是否在此子图右侧放 colorbar (只在每一行的最后一列或最后一个子图展示)
    is_cbar_col = (i % cols == cols - 1) or (i == n_targets - 1)
    
    # 绘制子图热图 (以浅灰展示未达显著或无互作的格子)
    sns.heatmap( 
        pivot_df, 
        cmap="Blues", 
        annot=True, 
        fmt=".2f", 
        linewidths=1.2, 
        linecolor='white', 
        vmin=vmin_global, 
        vmax=vmax_global, 
        cbar=is_cbar_col,
        cbar_kws={'shrink': 0.75, 'label': f'Mean Strength ({score_col})'} if is_cbar_col else None,
        mask=pivot_df.isnull(),
        ax=ax
    ) 
    
    # 样式美化：将缺失数据的背景设为极浅灰（学术规范）
    ax.set_facecolor('#EEEEEE') 
    for _, spine in ax.spines.items(): 
        spine.set_visible(True) 
        spine.set_color('#333333') 
        spine.set_linewidth(1.1) 
        
    ax.set_title(f"Target: {target_ec}", fontsize=11.5, fontweight='bold', pad=10) 
    ax.set_xlabel("Time Point", fontweight='bold', fontsize=10) 
    ax.tick_params(axis='x', rotation=0, labelsize=10.5)
    
    # 仅最左列保留 Y 轴通路标签，保持图面整洁
    if i % cols == 0: 
        ax.set_ylabel("Signaling Family", fontweight='bold', fontsize=10.5) 
        ax.tick_params(axis='y', rotation=0, labelsize=9.5) 
    else: 
        ax.set_ylabel("") 
        ax.set_yticks([]) 

# 隐藏多余的空白子图
for j in range(n_targets, len(axes)): 
    fig.delaxes(axes[j]) 

# 全局标题（明确声明 FDR < 0.05 质控）
fig.suptitle(
    "FCN1+MP Macro-Pathway Signals to Distinct EC Subpopulations Over Time\n(Benjamini–Hochberg FDR < 0.05 Core Set)", 
    fontsize=14.5, 
    fontweight='bold', 
    y=1.02
) 

plt.tight_layout() 
plt.savefig("MacroPathway_FCN1MP_to_EachEC_Grid_FDR.pdf", dpi=300, bbox_inches="tight") 
plt.savefig("MacroPathway_FCN1MP_to_EachEC_Grid_FDR.png", dpi=300, bbox_inches="tight") 
plt.show() 

print("-> 已生成基于 FDR 校正的分面板对比热图：MacroPathway_FCN1MP_to_EachEC_Grid_FDR.pdf")

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches

# =========================================================
# 0. 全局字体与排版设置
# =========================================================
plt.rcParams['font.family'] = 'sans-serif' 
plt.rcParams['font.sans-serif'] = ['Arial', 'Helvetica', 'DejaVu Sans'] 

# =========================================================
# 1. 自动适配打分列并严格基于 FDR < 0.05 筛选
# =========================================================
print("正在绘制基于 FDR < 0.05 的时空动态高分辨率气泡图...")

candidate_score_cols = ["lrscore", "lr_means", "mean", "expr_prod"]
score_col = None
for col in candidate_score_cols:
    if col in df_fcn1_to_ec.columns:
        score_col = col
        break

if score_col is None:
    raise ValueError(f"未找到可用的通讯强度列，请检查 df_fcn1_to_ec 的列名: {list(df_fcn1_to_ec.columns)}")

print(f"-> 气泡图采用的通讯强度指标: '{score_col}'")

# 必须只保留通过 Benjamini-Hochberg 校正的数据 (FDR < 0.05)
df_core = df_fcn1_to_ec[df_fcn1_to_ec['cellphone_padj'] < 0.05].copy() 
df_core["LR_Pair"] = df_core["ligand_complex"].astype(str) + " → " + df_core["receptor_complex"].astype(str) 

# 核心通路定义（剔除假阳性代谢类通路）
ordered_families = [ 
    "TGF-b / BMP Signaling", 
    "Semaphorin & Guidance", 
    "Growth Factor / Angiogenesis", 
    "Adhesion / ECM / Matrix", 
    "Chemokine / Cytokine", 
    "Complement System"
] 
df_core = df_core[df_core["Signaling Family"].isin(ordered_families)].copy()

# 每个家族挑选强度最高的前 4 对高置信互作
top_lrs = df_core.groupby(["Signaling Family", "LR_Pair"], observed=False)[score_col].max().reset_index() 
top_lrs = top_lrs.sort_values(['Signaling Family', score_col], ascending=[True, False]) 
keep_lrs = top_lrs.groupby("Signaling Family").head(4)["LR_Pair"].tolist() 

plot_df = df_core[df_core["LR_Pair"].isin(keep_lrs)].copy() 

# =========================================================
# 2. 生成 Y 轴坐标与家族分组排序
# =========================================================
plot_df['Family_Cat'] = pd.Categorical(plot_df['Signaling Family'], categories=ordered_families, ordered=True) 
plot_df = plot_df.sort_values(by=["Family_Cat", "LR_Pair"], ascending=[False, False]) 

y_labels = plot_df["LR_Pair"].unique() 
y_mapping = {label: i for i, label in enumerate(y_labels)} 
plot_df["Y_Pos"] = plot_df["LR_Pair"].map(y_mapping) 

# =========================================================
# 3. 生成 X 轴坐标 (Target EC | Time Point)
# =========================================================
targets = sorted(plot_df["target"].unique()) 
times = ["D14", "D33", "D61"] 

x_labels = [] 
x_mapping = {} 
idx = 0
for tgt in targets: 
    for t in times: 
        x_labels.append(f"{tgt} | {t}") 
        x_mapping[(tgt, t)] = idx
        idx += 1

plot_df["X_Pos"] = plot_df.apply(lambda row: x_mapping.get((row["target"], row["time_point"]), -1), axis=1) 
plot_df = plot_df[plot_df["X_Pos"] != -1] 

# 核心升级：基于 FDR (cellphone_padj) 计算气泡尺寸
# 气泡大小正比于 -log10(FDR)，严格反映统计显著性
plot_df["cellphone_padj"] = np.clip(plot_df["cellphone_padj"].astype(float), 1e-6, 0.05)
plot_df["-log10(FDR)"] = -np.log10(plot_df["cellphone_padj"]) 

SIZE_SCALE = 65
SIZE_BASE = 35
plot_df["bubble_size"] = plot_df["-log10(FDR)"] * SIZE_SCALE + SIZE_BASE

# =========================================================
# 4. 绘制高颜值时空气泡图
# =========================================================
n_y = len(y_labels)
fig_height = max(8.5, n_y * 0.45 + 3.0)
fig, ax = plt.subplots(figsize=(15.5, fig_height))  

# 给右侧留出 26% 的空间放文字、FDR 图例和独立 Colorbar
fig.subplots_adjust(right=0.74, top=0.90, bottom=0.10) 

# 绘制交替底色与通路分割线
shade = False
for family in ordered_families: 
    sub = plot_df[plot_df["Family_Cat"] == family] 
    if sub.empty: 
        continue
        
    ymin, ymax = sub["Y_Pos"].min() - 0.5, sub["Y_Pos"].max() + 0.5
    if shade: 
        ax.axhspan(ymin, ymax, facecolor='#F4F6F9', edgecolor='none', zorder=0, alpha=0.85) 
    shade = not shade
    
    # 家族名称紧贴右侧
    ax.text(x=len(x_labels) + 0.25, y=(ymin+ymax)/2, s=family, 
            va='center', ha='left', fontweight='bold', color='#1A365D', fontsize=11, clip_on=False) 
    ax.axhline(ymax, color='#D5D8DC', lw=0.7, zorder=1) 

ax.axhline(-0.5, color='#D5D8DC', lw=0.7, zorder=1) 

# 绘制气泡散点
scatter = ax.scatter( 
    plot_df["X_Pos"], 
    plot_df["Y_Pos"], 
    s=plot_df["bubble_size"], 
    c=plot_df[score_col], 
    cmap="Blues",                      
    edgecolors="#0B3C68",      
    linewidths=0.85,         
    alpha=0.95, 
    zorder=3
) 

# 纵向虚线分隔各个 EC 亚群
for i in range(1, len(targets)): 
    ax.axvline(i * 3 - 0.5, color='#BDC3C7', lw=1.1, linestyle='--', alpha=0.6, zorder=1) 

# 顶部 Target EC 亚群分组横线与标签
y_top_line = n_y + 0.1 
y_top_text = n_y + 0.45 

for i, tgt in enumerate(targets): 
    start_x = i * 3
    end_x = start_x + 2
    center_x = (start_x + end_x) / 2
    
    ax.plot([start_x, end_x], [y_top_line, y_top_line], color='#2C3E50', lw=1.5, zorder=4, clip_on=False) 
    ax.plot([start_x, start_x], [y_top_line-0.12, y_top_line], color='#2C3E50', lw=1.5, zorder=4, clip_on=False) 
    ax.plot([end_x, end_x], [y_top_line-0.12, y_top_line], color='#2C3E50', lw=1.5, zorder=4, clip_on=False) 
    
    ax.text(x=center_x, y=y_top_text, s=tgt, 
            ha='center', va='bottom', fontsize=10.5, fontweight='bold', color='#1A5276', clip_on=False) 

# 黑色圆角边框高亮每个 EC 亚群中最强的互作
for tgt in targets: 
    tgt_df = plot_df[plot_df["target"] == tgt] 
    if tgt_df.empty: 
        continue
    max_idx = tgt_df[score_col].idxmax() 
    max_row = tgt_df.loc[max_idx] 
    max_x, max_y = max_row["X_Pos"], max_row["Y_Pos"] 
    
    highlight_box = patches.FancyBboxPatch( 
        (max_x - 0.42, max_y - 0.42), 0.84, 0.84, 
        boxstyle="round,pad=0.02,rounding_size=0.06", 
        linewidth=1.6, edgecolor='#17202A', facecolor='none', zorder=5                      
    ) 
    ax.add_patch(highlight_box) 

# =========================================================
# 5. 坐标轴美化
# =========================================================
ax.set_yticks(range(len(y_labels))) 
ax.set_yticklabels(y_labels, fontsize=10) 

ax.set_xticks(range(len(x_labels))) 
clean_x_labels = [label.split(" | ")[1] for label in x_labels] 
ax.set_xticklabels(clean_x_labels, rotation=45, ha='right', fontsize=10, fontweight='bold') 

ax.spines['top'].set_visible(False) 
ax.spines['right'].set_visible(False) 
ax.spines['bottom'].set_visible(False) 
ax.spines['left'].set_linewidth(1.2) 

ax.set_xlim(-0.5, len(x_labels) - 0.5) 
ax.set_ylim(-0.8, n_y + 1.2) 

# =========================================================
# 6. 右侧图例终极排版 (FDR 图例 & 独立 Colorbar)
# =========================================================
# 6.1 FDR (q-value) 显著性梯度图例
fdr_ticks = [0.05, 0.01, 0.001] 
fdr_log_s = [-np.log10(p) for p in fdr_ticks] 
legend_sizes = [lp * SIZE_SCALE + SIZE_BASE for lp in fdr_log_s] 

legend_markers = [plt.scatter([], [], s=s, c='#85C1E9', edgecolors='#0B3C68', lw=0.85) for s in legend_sizes] 
leg = ax.legend(
    legend_markers, 
    ['< 0.05', '< 0.01', '< 0.001'], 
    title='FDR (q-value)', 
    loc='upper left', 
    bbox_to_anchor=(1.36, 0.96), 
    frameon=False, 
    labelspacing=1.3, 
    title_fontsize=11,
    fontsize=10
) 
leg._legend_box.align = "left" 

# 6.2 独立垂直色条 Colorbar
cbar_ax = fig.add_axes([0.91, 0.28, 0.012, 0.32]) 
cbar = plt.colorbar(scatter, cax=cbar_ax) 
cbar.set_label(f'Communication Intensity ({score_col})', rotation=270, labelpad=18, fontweight='bold', fontsize=10.5) 
cbar.outline.set_linewidth(0.8) 
cbar.ax.tick_params(labelsize=9.5) 

# =========================================================
# 7. 标题设置
# =========================================================
plt.text(0.38, 1.08, "Spatiotemporal Dynamics of FCN1+MP $\\rightarrow$ Endothelial Cell Signaling", 
         fontsize=15, fontweight='bold', ha='center', transform=ax.transAxes) 
plt.text(0.38, 1.035, "(All interactions strictly thresholded at Benjamini–Hochberg FDR < 0.05; black boxes denote top hits)", 
         fontsize=11, style='italic', color='#566573', ha='center', transform=ax.transAxes) 

# 保存高质量矢量图
plt.savefig("AllFamilies_FCN1MP_to_EC_FDR_Spatiotemporal.pdf", dpi=300, bbox_inches="tight") 
plt.savefig("AllFamilies_FCN1MP_to_EC_FDR_Spatiotemporal.png", dpi=300, bbox_inches="tight") 
plt.show() 

print("-> 已生成基于 FDR 严格校正的终极时空气泡图：AllFamilies_FCN1MP_to_EC_FDR_Spatiotemporal.pdf")

In [ ]:
import scanpy as sc
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

# =========================================================
# 0. 全局样式设置
# =========================================================
sns.set_theme(style="ticks") 
plt.rcParams['font.sans-serif'] = ['Arial', 'Helvetica', 'DejaVu Sans'] 
plt.rcParams['axes.unicode_minus'] = False

# =========================================================
# 1. 选取经过 FDR 严格验证的核心王牌配受体对 (替换掉有争议的 QDPR/DYSF)
# =========================================================
# 4 对极具生物学解释力的高置信轴：
pairs = [ 
    ("VEGFD", "FLT1", "Angiogenesis"), 
    ("ICAM1", "CAV1", "Adhesion / Transmigration"), 
    ("TGFB1", "ENG", "ECM & Remodeling"), 
    ("SEMA3B", "NRP1", "Endothelial Guidance") 
] 

# 提取这 8 个基因（自动处理基因名大小写与存在性）
available_genes = adata_cross.var_names.tolist()
genes_to_extract = []
for lig, rec, _ in pairs:
    genes_to_extract.extend([lig, rec])

# 细胞亚群有序排列 (前 3 个为人 MP，后 6 个为猪 EC)
ordered_cell_types = [ 
    'FCN1+MP', 'MARCO+MP', 'MP1', 
    'ECM-remodeling Arterial EC', 'Arterial EC', 
    'Capillary EC_1', 'Capillary EC_2', 
    'Glomerular EC', 'Venous EC' 
] 
# 过滤掉 adata 中实际不存在的类型
ordered_cell_types = [c for c in ordered_cell_types if c in adata_cross.obs['celltype_for_liana'].unique()]

# =========================================================
# 2. 计算各亚群中的平均表达量并进行 Min-Max 归一化
# =========================================================
mean_expr = pd.DataFrame(index=ordered_cell_types) 

for gene in set(genes_to_extract): 
    if gene in adata_cross.var_names:
        expr = adata_cross[:, gene].X
        if hasattr(expr, "toarray"):
            expr = expr.toarray().flatten()
        else:
            expr = np.asarray(expr).flatten()
            
        df_tmp = pd.DataFrame({'Expr': expr, 'Celltype': adata_cross.obs['celltype_for_liana']}) 
        cell_means = df_tmp.groupby('Celltype')['Expr'].mean().reindex(ordered_cell_types).fillna(0) 
        
        # 归一化到 0 - 1
        denom = (cell_means.max() - cell_means.min())
        mean_expr[gene] = (cell_means - cell_means.min()) / (denom + 1e-8) if denom > 0 else 0.0
    else:
        print(f"Warning: 基因 {gene} 不在 adata 中，用 0 填充")
        mean_expr[gene] = 0.0

# =========================================================
# 3. 创建子图 (1行4列，带物种背景分界与极简美化)
# =========================================================
fig, axes = plt.subplots(1, 4, figsize=(19, 4.6), sharey=True) 

# 配色：配体（暖红珊瑚色），受体（科技深蓝色）
colors = {'Ligand': '#D9534F', 'Receptor': '#2E86C1'} 

# 计算 MP 和 EC 的分界线位置
n_mp = len([c for c in ordered_cell_types if 'MP' in c.upper()])
split_x = n_mp - 0.5

for i, (ligand, receptor, title) in enumerate(pairs): 
    ax = axes[i] 
    
    # 3.1 背景阴影区分物种与细胞大类
    # MP 区域 (人类浸润免疫)
    ax.axvspan(-0.5, split_x, color='#FADBD8', alpha=0.25, zorder=0)
    # EC 区域 (供体猪移植物血管)
    ax.axvspan(split_x, len(ordered_cell_types) - 0.5, color='#EBF5FB', alpha=0.35, zorder=0)
    
    # 分界线
    ax.axvline(split_x, color='#BDC3C7', linestyle=':', lw=1.2, zorder=1)
    
    # 顶部添加物种/大类指示文字 (仅在第一个图或每个图加)
    if i == 0:
        ax.text(split_x / 2, 1.08, "Human Infiltrates\n(Sender)", 
                ha='center', va='bottom', fontsize=8.5, fontweight='bold', color='#78281F', transform=ax.get_xaxis_transform())
        ax.text((split_x + len(ordered_cell_types) - 0.5) / 2, 1.08, "Pig Graft Endothelium\n(Receiver)", 
                ha='center', va='bottom', fontsize=8.5, fontweight='bold', color='#1B4F72', transform=ax.get_xaxis_transform())

    # 3.2 绘制折线
    ax.plot(ordered_cell_types, mean_expr[ligand], marker='o', color=colors['Ligand'], 
            label=f"Ligand: {ligand}", lw=2.2, markersize=7.5, markeredgecolor='white', markeredgewidth=1.2, zorder=3) 
    ax.plot(ordered_cell_types, mean_expr[receptor], marker='s', color=colors['Receptor'], 
            label=f"Receptor: {receptor}", lw=2.2, markersize=7.5, markeredgecolor='white', markeredgewidth=1.2, zorder=3) 
    
    # 标题设计
    ax.set_title(f"{title}\n({ligand} $\\rightarrow$ {receptor})", fontsize=11.5, fontweight='bold', pad=12, color='#1A252C') 
    
    # X 轴美化
    ax.set_xticks(range(len(ordered_cell_types))) 
    ax.set_xticklabels(ordered_cell_types, rotation=45, ha='right', fontsize=9.5, color='#2C3E50', fontweight='bold') 
    
    # Y 轴与网格线
    ax.grid(axis='y', linestyle='--', color='#E5E7E9', alpha=0.8, zorder=1) 
    ax.set_axisbelow(True)
    ax.set_ylim(-0.05, 1.15)
    
    ax.spines['left'].set_color('#7F8C8D') 
    ax.spines['bottom'].set_color('#7F8C8D') 
    ax.tick_params(colors='#2C3E50', labelsize=9.5) 
    
    if i == 0: 
        ax.set_ylabel("Normalized Mean Expression (0–1)", fontsize=11, fontweight='bold', color='#2C3E50') 
    
    # 图例设置 (左上角，干净无框)
    ax.legend(loc='upper right', frameon=False, fontsize=9.5, handletextpad=0.4) 

# 去除多余边框
sns.despine() 

plt.suptitle("Cell-type Specific Expression Profiles of Verified Cross-species Signaling Axes", 
             fontsize=14, fontweight='bold', y=1.06)

plt.tight_layout() 

# 导出为高分论文级矢量图
plt.savefig("LR_Pairs_Line_Profile_Polished_FDR.pdf", dpi=300, bbox_inches='tight') 
plt.savefig("LR_Pairs_Line_Profile_Polished_FDR.png", dpi=300, bbox_inches='tight') 
plt.show() 

print("-> 已生成无争议、高置信的配受体亚群表达谱图：LR_Pairs_Line_Profile_Polished_FDR.pdf")